# CHB-MIT Low-Latency Seizure Decoder

Causal preprocessing -> features -> patient calibration -> 1D-TCN -> streaming fixed-point port.

**Status: end-to-end deployable.** Patient-specific causal TCN at **sensitivity
0.86-1.00, 0.4-1.8 FA/h, 5-21 s latency** across six subjects, beating
patient-calibrated logreg on five. Streaming fixed-point inference in
**58,784 MACs/timestep and ~82 KB**; Q15 costs **zero** detection performance.

Cross-subject transfer fails for all four model classes tried. That is the
finding; patient calibration is the answer.

> ### After any runtime disconnect
> Paste **`RESTORE_CELL.py`** first. ~30 s. Do **not** Run All.

| section | what it does | state |
|---|---|---|
| 0-8 | Data layer: acquisition -> features | done, 26/26 shards |
| 9 | Evaluation harness | done, self-tested |
| 10 | LOSO baselines | LogReg 23/23, RF diagnostic |
| 11 | Cross-subject diagnosis | done - 5 experiments |
| 12 | Patient calibration curve | done |
| 13 | **Figures** | 2/6 rendered |
| 14 | 1D-TCN | done - 6 subjects, 3 seeds |
| 15 | Streaming fixed-point port | done and validated |
| 16 | Roadmap | - |

```
686 EDFs | 676 annotated | 198 seizures | 3.23 h ictal
3,480,880 windows | 11,809 ictal (0.339%) | 23 subject groups
```

Guttag, J. (2010). *CHB-MIT Scalp EEG Database* (v1.0.0). PhysioNet. RRID:SCR_007345.

---
# 0. Setup and definitions

Everything below section 0 depends on these cells and nothing else. After any
disconnect, run 0.0 through 0.5 in order and every later section works again.

0.0 also detects the environment, so the same notebook runs in Colab, locally,
or on a cluster without edits.

### 0.0 Environment and session check - run this first

Detects Colab, local, or cluster and derives every path from four variables.
There are no hardcoded paths anywhere else in the notebook.

Point it at your data with an environment variable, no code change:

```bash
export CHBMIT_BASE=/scratch/$USER/chbmit
```

or set `CHBMIT_ROOT`, `CHBMIT_OUT` and `CHBMIT_ARCHIVE` individually if your
cluster splits scratch from home.

It also reports what survived: 0 EDFs means the raw corpus is gone (re-sync in
section 1.1), 0 shards means the feature tables are gone (restore from ARCHIVE,
or re-extract in section 8).

**Restore order after any disconnect:** this cell, then 0.1 install if the
environment is new, then 0.2 imports, 0.3 constants, 0.4 parser, 0.5
`summarized`. That last one is easy to forget, and an empty dict silently makes
every record look unannotated.

In [ ]:
# Environment configuration. Runs in Colab, locally, or on a cluster.
# Every path below is derived from these four variables, so the notebook has no
# hardcoded /content/ anywhere. Override with environment variables if your
# layout differs.

import os, sys, glob

IN_COLAB = 'google.colab' in sys.modules or os.path.isdir('/content')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT    = f'{ROOT}'
    OUT_DIR = f'{OUT_DIR}'
    ARCHIVE = f'{ARCHIVE}'
else:
    BASE    = os.environ.get('CHBMIT_BASE', os.path.expanduser('~/chbmit'))
    ROOT    = os.environ.get('CHBMIT_ROOT',    f'{BASE}/raw')
    OUT_DIR = os.environ.get('CHBMIT_OUT',     f'{BASE}/data')
    ARCHIVE = os.environ.get('CHBMIT_ARCHIVE', f'{BASE}/archive')

FEATURES = f'{OUT_DIR}/features'
FIGURES  = f'{OUT_DIR}/figures'
CODE     = f'{ARCHIVE}/code'
for d in (OUT_DIR, FEATURES, FIGURES, ARCHIVE, CODE):
    os.makedirs(d, exist_ok=True)

n_edf   = len(glob.glob(f'{ROOT}/chb*/*.edf'))
n_shard = len(glob.glob(f'{FEATURES}/*.parquet'))

print(f"env       {'colab' if IN_COLAB else 'local / cluster'}")
print(f"ROOT      {ROOT}")
print(f"FEATURES  {FEATURES}")
print(f"ARCHIVE   {ARCHIVE}")
print()
print(f'{n_edf:4d} EDFs        (expect 686; 0 means re-sync from S3, section 1.1)')
print(f'{n_shard:4d} shards      (expect 26; 0 means restore from ARCHIVE or re-extract)')
print(f'manifest    {os.path.exists(f"{OUT_DIR}/record_manifest.csv")}')

### 0.1 Install

One cell, then **Runtime -> Restart session** if Colab asks.

In [ ]:
!pip install -q awscli mne PyWavelets pyarrow

### 0.2 Imports

In [ ]:
# --- stdlib ---
import os, re, glob, time, gc, json, shutil
from collections import Counter, defaultdict

# --- scientific ---
import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view
from scipy import signal
import matplotlib.pyplot as plt

# --- domain ---
import mne, pywt
mne.set_log_level('ERROR')

import scipy
print(f'mne {mne.__version__} | pywt {pywt.__version__} | '
      f'numpy {np.__version__} | scipy {scipy.__version__} | pandas {pd.__version__}')

### 0.3 Channel constants

`CANONICAL` index order is **fixed and load-bearing** - downstream code indexes
channels by integer position, so reordering this list silently corrupts
everything after it.

Dropped from the reference 23-channel montage:
- `T7-FT9`, `FT9-FT10`, `FT10-T8` - FT9/FT10 are not standard 10-20 positions and
  the chain is inconsistently present, which would make the feature vector's
  shape subject-dependent
- `P7-T7` - polarity inverse of `T7-P7`; including both gives a collinear pair

Two normalization quirks, both discovered empirically in Section 2:
- MNE appends `-0`/`-1` to duplicate names (`T8-P8` appears twice in the
  23-channel header). The suffix strip must not eat `FT9-FT10`.
- `01` is a zero-one **typo** for `O1` in the chb12_28/_29 headers.

In [ ]:
CANONICAL = ['FP1-F7','F7-T7','T7-P7','P7-O1',    # left temporal
             'FP1-F3','F3-C3','C3-P3','P3-O1',    # left parasagittal
             'FP2-F4','F4-C4','C4-P4','P4-O2',    # right parasagittal
             'FP2-F8','F8-T8','T8-P8','P8-O2',    # right temporal
             'FZ-CZ','CZ-PZ']                     # midline

ALIASES = {'01': 'O1'}   # zero-one typo in the chb12_28/_29 EDF headers

def norm(n):
    """Uppercase, strip whitespace, drop MNE's duplicate-run suffix, apply aliases."""
    n = n.strip().upper().replace(' ', '')
    m = re.search(r'-(\d+)$', n)
    if m and n[:m.start()].count('-') >= 1:      # 'T8-P8-0' -> 'T8-P8', but 'FT9-FT10' survives
        n = n[:m.start()]
    return ALIASES.get(n, n)

assert norm('T8-P8-0') == 'T8-P8'
assert norm('FT9-FT10') == 'FT9-FT10'
assert norm('01') == 'O1'
print(f'{len(CANONICAL)} canonical channels, normalizer OK')

# Column names that are metadata, not features. Used by sections 9, 10, 12, 14
# and 15, so it is defined once here rather than at first use.
META_COLS = ('y','subject_group','subject','record','seizure_event','t_start',
             'derivation','noisy')

### 0.4 Summary parser

Regex-anchored on the **numeric value**, not on a `': '` delimiter. Section 3.4
has the forensics: `chb12_27` mixes one- and two-space separators *within a
single record*, and a delimiter split silently dropped 12 seizures.

Validates parsed count against the declared `Number of Seizures in File` and
**raises** on mismatch. Warnings are not enough - they scroll off the top of the
Colab output and a corrupted annotation set propagated into a signal-quality
measurement that looked significant and was not.

In [ ]:
import re

RE_FILE  = re.compile(r'^File\s+Name:\s*(\S+)', re.I)
RE_NSZ   = re.compile(r'^Number\s+of\s+Seizures\s+in\s+File:\s*(\d+)', re.I)
RE_START = re.compile(r'^Seizure\s*\d*\s*Start\s+Time:\s*([\d.]+)\s*seconds', re.I)
RE_END   = re.compile(r'^Seizure\s*\d*\s*End\s+Time:\s*([\d.]+)\s*seconds', re.I)


class EDFFileSummary:
    def __init__(self, filename):
        self.filename = filename
        self.n_declared = 0
        self.seizures = []


def parse_summary(path):
    summaries, cur, pending = {}, None, None

    for lineno, raw in enumerate(open(path), 1):
        line = raw.strip()
        if not line:
            continue

        if m := RE_FILE.match(line):
            if pending is not None:
                raise ValueError(f'{path}:{lineno} unterminated seizure in {cur.filename}')
            cur = EDFFileSummary(m.group(1))
            summaries[cur.filename] = cur

        elif cur is None:
            continue                                  # header before first record

        elif m := RE_NSZ.match(line):
            cur.n_declared = int(m.group(1))

        elif m := RE_START.match(line):
            if pending is not None:
                raise ValueError(f'{path}:{lineno} two starts in a row, {cur.filename}')
            pending = float(m.group(1))

        elif m := RE_END.match(line):
            end = float(m.group(1))
            if pending is None:
                raise ValueError(f'{path}:{lineno} end with no start, {cur.filename}')
            if end < pending:
                raise ValueError(f'{path}:{lineno} end {end} < start {pending}')
            cur.seizures.append((pending, end))
            pending = None
        # Channel/montage-change lines fall through, ignored

    if pending is not None:
        raise ValueError(f'{path}: file ends mid-seizure in {cur.filename}')

    for rec in summaries.values():
        if len(rec.seizures) != rec.n_declared:
            raise ValueError(f'{rec.filename}: declared {rec.n_declared}, '
                             f'parsed {len(rec.seizures)}')
    return summaries

### 0.6 Rebuild `summarized` - do not skip

`summarized` is a plain dict and vanishes on restart. An empty dict makes
`window_record` treat **every** record as unannotated, which prints "no usable
records" and silently produces empty shards. This cost a full chb17 extraction.

The assertion below turns that silent failure into a stop.

In [ ]:
summarized = {}
for s in sorted(glob.glob(f'{ROOT}/chb*/chb*-summary.txt')):
    summarized.update(parse_summary(s))

assert len(summarized) == 676, f'expected 676 records, got {len(summarized)}'
print(len(summarized), 'records,',
      sum(len(r.seizures) for r in summarized.values()), 'seizures')

# the 10 unannotated chb24 records are the ONLY legitimate summary misses
KNOWN_UNANNOTATED = {f'chb24_{n:02d}.edf' for n in
                     (2, 5, 8, 10, 12, 16, 18, 19, 20, 22)}

---
# 1. Acquisition and integrity

**Result: 686 EDF files, 45.8 GB, all byte-exact against the S3 manifest.**

### 1.1 Download

S3 mirror rather than HTTP recursive fetch. A prior `wget -r` against
physionet.org sustained ~161 KB/s - roughly three days for the full corpus.
`aws s3 sync` is resumable, so re-running it after an interruption is safe.

In [ ]:
!pip install awscli
!aws s3 sync --no-sign-request s3://physionet-open/chbmit/1.0.0/ {ROOT}/

### 1.2 Size heuristic - SUPERSEDED, kept as a record

Flagged 5 files under 15 MB. **All false positives.** Hardware gaps between
consecutively-numbered records mean some CHB-MIT files are legitimately short -
`chb24_22.edf` is ~18 minutes, not a truncated hour. The threshold is a
reasonable smell test for a corpus of uniform 1-hour files and a bad one here.

In [ ]:
import os, glob
edfs = glob.glob(f'{ROOT}/chb*/*.edf')
small = [f for f in edfs if os.path.getsize(f) < 15e6]
print(f"{len(edfs)} EDFs, {sum(os.path.getsize(f) for f in edfs)/1e9:.1f} GB")
print(f"suspiciously small: {small}" if small else "no partial files")

### 1.3 EDF header consistency - SUPERSEDED, kept as a record

Flagged `chb17b_69.edf` as truncated. **False positive, and the label was
backwards** - the file is 256 bytes *larger* than predicted, not smaller.

This checker recomputes header length as `256 * (ns + 1)` instead of reading the
declared value at header bytes 184:192. See 1.5.

In [ ]:
import glob, os

def edf_expected_size(path):
    with open(path, 'rb') as f:
        head = f.read(256)
        n_records = int(head[236:244]); ns = int(head[252:256])
        f.seek(256 + ns * 216)
        spr = sum(int(f.read(8)) for _ in range(ns))
    if n_records < 0:
        return None, ns, n_records
    return 256 * (ns + 1) + n_records * spr * 2, ns, n_records

for p in sorted(glob.glob(f'{ROOT}/chb*/*.edf')):
    exp, ns, nrec = edf_expected_size(p)
    act = os.path.getsize(p)
    if exp is None:
        print(f'{os.path.basename(p)}: header declares unknown length, skipped')
    elif act != exp:
        print(f'TRUNCATED {os.path.basename(p)}: {act:,} vs {exp:,} declared')
print('header check complete')

### 1.4 S3 manifest diff - AUTHORITATIVE

Byte-exact size comparison against the remote objects. This supersedes 1.2 and
1.3 and is the check to trust.

**686/686 complete, 0 size mismatches, 0 missing.**

In [ ]:
!aws s3 ls --no-sign-request --recursive s3://physionet-open/chbmit/1.0.0/ > {OUT_DIR}/s3_manifest.txt

import os
remote = {}
for line in open(f'{OUT_DIR}/s3_manifest.txt'):
    parts = line.split()
    if len(parts) >= 4 and parts[3].endswith('.edf'):
        remote[parts[3].split('chbmit/1.0.0/')[-1]] = int(parts[2])

bad, missing = [], []
for rel, size in remote.items():
    local = f'{ROOT}/{rel}'
    if not os.path.exists(local):
        missing.append(rel)
    elif os.path.getsize(local) != size:
        bad.append((rel, os.path.getsize(local), size))

print(f'remote EDFs: {len(remote)}   local complete: {len(remote)-len(bad)-len(missing)}')
print(f'size mismatch: {bad}' if bad else 'no size mismatches')
print(f'missing: {missing}' if missing else 'nothing missing')

### 1.5 chb17b_69 - resolved, benign

Header declares 7,424 bytes = 256 x (28+1). Plus 3,600 records x 28 ch x 256
samples x 2 bytes = 51,617,024. File is 51,617,280, so **256 trailing bytes of
padding**. A data record here is 14,336 bytes, so 256 bytes cannot be a partial
record. MNE reads exactly 60.0 minutes.

In [ ]:
!pip install mne

import mne

p = f'{ROOT}/chb17/chb17b_69.edf'
with open(p, 'rb') as f:
    h = f.read(256)
print('declared header bytes:', int(h[184:192]), ' ns:', int(h[252:256]),
      ' -> 256*(ns+1) =', 256 * (int(h[252:256]) + 1))

raw = mne.io.read_raw_edf(p, preload=False, verbose='ERROR')
print(f'{raw.n_times / raw.info["sfreq"] / 60:.1f} min, {len(raw.ch_names)} channels, {raw.info["sfreq"]} Hz')

---
# 2. Montage survey

**Result: 12 distinct montages across 686 files, but 683/686 (99.6%) expose all
18 canonical channels.** The variants differ almost entirely in dummy (`--N`)
and non-canonical channels, not in the 18 that matter.

This is the empirical case for **selecting by name** rather than **dropping by
name**: extras like `--0`..`--4` and `EKG1-CHIN` become irrelevant instead of
something to enumerate.

### 2.1 Reference montage (chb01)

23 channels, 256 Hz, exactly 60.0 min. Note `T8-P8-0` / `T8-P8-1` and `P7-T7`.

In [ ]:
import mne
raw = mne.io.read_raw_edf(f'{ROOT}/chb01/chb01_03.edf', preload=False, verbose='ERROR')
print(len(raw.ch_names), raw.info['sfreq'], raw.n_times / raw.info['sfreq'] / 60)
print(raw.ch_names)

### 2.2 Montage variation across the corpus

`chb17b_69` has 28 channels but is missing **nothing** - five `--N` dummies.
`chb12_27` is missing **all 23**: it is CS2-referential, not bipolar.

In [ ]:
import mne, glob

ref = mne.io.read_raw_edf(f'{ROOT}/chb01/chb01_03.edf',
                          preload=False, verbose='ERROR')
chb01_names = ref.ch_names
print(f"reference chb01_03: {len(chb01_names)} ch")

for p in [f'{ROOT}/chb17/chb17b_69.edf',
          f'{ROOT}/chb12/chb12_27.edf']:
    r = mne.io.read_raw_edf(p, preload=False, verbose='ERROR')
    print(f"\n{p.split('/')[-1]}: {len(r.ch_names)} ch")
    print("  extra:  ", sorted(set(r.ch_names) - set(chb01_names)))
    print("  missing:", sorted(set(chb01_names) - set(r.ch_names)))

    from collections import Counter

counts, montages = Counter(), {}
for p in sorted(glob.glob(f'{ROOT}/chb*/*.edf')):
    r = mne.io.read_raw_edf(p, preload=False, verbose='ERROR')
    counts[len(r.ch_names)] += 1
    montages.setdefault(tuple(r.ch_names), []).append(p.split('/')[-1])

print("channel counts:", dict(counts))
print(f"distinct montages: {len(montages)}")
for m, files in sorted(montages.items(), key=lambda kv: -len(kv[1])):
    print(f"  {len(files):3d} files, {len(m):2d} ch  e.g. {files[0]}")


# For chb12 could either drop or re-derive

### 2.3 Canonical coverage - the number that matters

```
683 files | missing  0 canonical channels | all 26 case directories
  3 files | missing 18 canonical channels | chb12 only
```

In [ ]:
# CANONICAL and norm() come from section 0.4
import glob, mne
from collections import defaultdict
usable = defaultdict(list)
for p in sorted(glob.glob(f'{ROOT}/chb*/*.edf')):
    names = {norm(c) for c in mne.io.read_raw_edf(p, preload=False,
                                                  verbose='ERROR').ch_names}
    missing = [c for c in CANONICAL if c not in names]
    usable[tuple(missing)].append(p.split('/')[-1])

for miss, files in sorted(usable.items(), key=lambda kv: -len(kv[1])):
    subj = sorted({f.split('_')[0] for f in files})
    print(f"{len(files):3d} files | missing {len(miss):2d} | {subj}")
    if miss:
        print(f"      {list(miss)}")

### 2.4 Which chb12 files need re-derivation

`chb12_27`, `_28`, `_29`.

In [ ]:
# CANONICAL and norm() come from section 0.4
import glob, mne
for p in sorted(glob.glob(f'{ROOT}/chb12/*.edf')):
    names = {norm(c) for c in mne.io.read_raw_edf(p, preload=False,
                                                  verbose='ERROR').ch_names}
    if any(c not in names for c in CANONICAL):
        print('NEEDS RE-DERIVATION:', p.split('/')[-1])

### 2.5 Golden-file scan - NEGATIVE RESULT

Looked for any file containing both a bipolar channel **and** the two
referential channels it would be derived from, which would allow exact
sample-for-sample validation of the re-derivation.

**Nothing found.** Only statistical comparison is available, which caps
achievable confidence. Disclose this in the write-up.

In [ ]:
# CANONICAL and norm() come from section 0.4
import glob, mne
from collections import defaultdict

seen = {}
for p in sorted(glob.glob(f'{ROOT}/chb*/*.edf')):
    names = frozenset(norm(c) for c in
                      mne.io.read_raw_edf(p, preload=False, verbose='ERROR').ch_names)
    seen.setdefault(names, p)

for names, p in seen.items():
    both = []
    for pair in CANONICAL:
        a, b = pair.split('-')
        if pair in names and f'{a}-CS2' in names and f'{b}-CS2' in names:
            both.append(pair)
    if both:
        print(f'GOLDEN: {p.split("/")[-1]} has {len(both)} verifiable pairs -> {both[:4]}')

        #Nothings prints

### 2.6 The chb12_28/_29 montage

Bare electrode labels, no reference suffix - unlike `_27`'s explicit `-CS2`.
All 18 pairs are derivable **given the `01 -> O1` alias**: `O2` is spelled
correctly in the same file, so exactly one label was mistyped at recording time.
Without the alias, four pairs are blocked (`P7-O1`, `P3-O1`, and the chains they
terminate).

In [ ]:
for p in [f'{ROOT}/chb12/chb12_28.edf', f'{ROOT}/chb12/chb12_29.edf']:
    r = mne.io.read_raw_edf(p, preload=False, verbose='ERROR')
    print(f"\n{p.split('/')[-1]}:")
    print(sorted(c for c in r.ch_names if c.strip() not in ('-', '')))

---
# 3. Annotation layer

**Result: 686 records on disk, 676 annotated, 198 seizures, 3.23 h ictal.**
All 24 summary files parse without raising, including `chb24-summary.txt`,
which omits the `File Start Time` / `File End Time` lines present elsewhere.

The 3.23 h ictal total matches the published corpus description. The 198-seizure
count runs above the 182-185 commonly cited; since declared-equals-parsed is
asserted per record, this reflects what the annotation files contain rather than
an over-count. Footnote it rather than reconciling to a secondary source.

### 3.1 Parse every summary

Parser is defined in 0.4.

In [ ]:
import glob

total_sz = total_rec = 0
for s in sorted(glob.glob(f'{ROOT}/chb*/chb*-summary.txt')):
    recs = parse_summary(s)
    n = sum(len(r.seizures) for r in recs.values())
    total_sz, total_rec = total_sz + n, total_rec + len(recs)
    print(f'{s.split("/")[-1]:22s} {len(recs):3d} records  {n:3d} seizures')
print(f'\nTOTAL {total_rec} records, {total_sz} seizures')

### 3.2 Unannotated records - EXCLUDED

Ten `chb24` files appear on disk with no summary entry.

**Unannotated is not seizure-free.** Treating them as all-interictal would inject
unknown false negatives into training *and* inflate the interictal-hours
denominator, quietly depressing the reported false-alarm rate - a metric headed
for the write-up, so the denominator has to be defensible.

`chb24` keeps its 12 annotated records and 16 seizures, so the subject stays in
cross-validation. Cost is ~10 h of interictal time out of ~900.

Enforce in the loader: **no summary entry -> skip the record.** The version of
this bug that hurts is `summaries.get(name)` returning `None` and being treated
as an empty seizure list.

In [ ]:
import glob, os

summarized = set()
for s in sorted(glob.glob(f'{ROOT}/chb*/chb*-summary.txt')):
    summarized |= set(parse_summary(s))

on_disk = {os.path.basename(p) for p in glob.glob(f'{ROOT}/chb*/*.edf')}

print("on disk, not in any summary:", sorted(on_disk - summarized))
print("in summary, not on disk:    ", sorted(summarized - on_disk))

### 3.3 Record manifest

Every inclusion decision in one table: annotated or not, seizure count, ictal
seconds, file size. Add a `derivation` column once Section 5 classifies each
file, and this becomes the artifact a reviewer asks for.

Write it to Drive, not just `/content`.

In [ ]:
import os, glob, pandas as pd

os.makedirs(f'{OUT_DIR}', exist_ok=True)

summarized = {}
for s in sorted(glob.glob(f'{ROOT}/chb*/chb*-summary.txt')):
    summarized.update(parse_summary(s))

rows = []
for p in sorted(glob.glob(f'{ROOT}/chb*/*.edf')):
    name = os.path.basename(p)
    rec = summarized.get(name)
    rows.append({
        'record': name,
        'subject_dir': name.split('_')[0],
        'annotated': rec is not None,
        'n_seizures': len(rec.seizures) if rec else None,
        'ictal_sec': sum(e - s for s, e in rec.seizures) if rec else None,
        'size_mb': round(os.path.getsize(p) / 1e6, 1),
    })

man = pd.DataFrame(rows)
man.to_csv(f'{OUT_DIR}/record_manifest.csv', index=False)
print(f"{len(man)} records, {man.annotated.sum()} annotated, "
      f"{int(man.n_seizures.sum())} seizures, "
      f"{man.ictal_sec.sum()/3600:.2f} h ictal")
man[~man.annotated]

### 3.4 Whitespace forensics - why the parser is regex-anchored

`chb12_27` contains **both** separators within a single record:

```
Seizure 1 Start Time: 916 seconds     <- one space
Seizure 1 End Time:  951 seconds      <- two spaces
```

`line.split(': ')[1]` returns `' 951 seconds'`, and `.split(' ')[0]` on a leading
space yields `''`. This silently dropped all 6 seizures from `chb12_27` and all 6
from `chb12_29`, and the resulting corrupted annotations propagated into
Section 4's power comparison.

In [ ]:
!grep -n "Seizure" {ROOT}/chb12/chb12-summary.txt | sed -n '1,12p' | cat -A | head -12
!awk '/chb12_27/,/chb12_30/' {ROOT}/chb12/chb12-summary.txt | cat -A

---
# 4. chb12 re-derivation - RESOLVED

Three chb12 files are not bipolar, which invalidates the general claim that
CHB-MIT is uniformly bipolar.

| file | montage | seizures |
|---|---|---|
| `chb12_27` | referential to CS2 - `FP1-CS2`, `F7-CS2`, ... | 6 |
| `chb12_28` | bare electrode labels - `FP1`, `F7`, ... | 1 |
| `chb12_29` | bare electrode labels | 6 |

**13 seizures out of chb12's 40** - about a third of the patient, ~6.5% of the
corpus. Dropping was rejected on that basis.

Re-derivation is exact where the reference is shared:
`FP1-F7 = (FP1-CS2) - (F7-CS2)`, since CS2 cancels identically.

**Caveat:** for `_27` the shared reference is explicit in the labels, so the
subtraction is provably exact. For `_28`/`_29` the labels are bare and the
reference is *unstated* - a common reference is assumed. Very likely correct,
but recorded as an assumption.

> ### VERDICT: derivation is sound; the recordings are noisier
> Validated on four independent grounds (4.3, 4.5). No golden test file exists
> (2.5), so exact sample-for-sample validation is impossible - only spectral and
> statistical comparison. That caps confidence and is disclosed.
>
> **The conclusion does not depend on the n=3 group statistic.** The spectral
> shape argument in 4.5 settles it at n=1. The Cohen's *d* reported during
> development is meaningless (std = 0 with one sample) and is disregarded.

### 4.6 Signal-quality comparison helper

Masks ictal +/- 120 s before measuring in-band power, so the comparison
is between interictal segments only. Without the mask the measurement
includes seizure content and the result is meaningless.

**Requires `load_18` from section 5.** This section reads earlier than it
runs: the chb12 investigation is what motivated the three-mode loader, so
run section 5 before executing 4.6 and 4.7.

In [ ]:
# Spectral Resolution

import numpy as np, glob
from scipy import signal

sz = parse_summary(f'{ROOT}/chb12/chb12-summary.txt')

def clean_power(path, tag):
    try:
        X, _ = load_18(path)
    except ValueError as e:
        print(f"Warning: Skipping file {path} due to error in load_18: {e}")
        return None # Return None if channels cannot be derived
    n = X.shape[1]
    mask = np.ones(n, bool)
    for s0, s1 in sz[path.split('/')[-1]].seizures:      # drop ictal +/- 120 s
        mask[max(0, int((s0-120)*256)) : int((s1+120)*256)] = False
    X = X[:, mask]
    if X.shape[1] < 256*120:
        return None
    f, P = signal.welch(X, fs=256, nperseg=1024, axis=-1)
    return (10*np.log10(P[:, (f>=1)&(f<=40)].mean()),
            np.percentile(np.abs(X), 99)*1e6, tag)

### 4.1 Seizure burden of the affected files

In [ ]:
# Do the 3 dropped files contain seizures?
!grep -A4 -E "chb12_(27|28|29|32)" {ROOT}/chb12/chb12-summary.txt | head -40
!grep -c "Number of Seizures in File: [1-9]" {ROOT}/chb12/chb12-summary.txt
!grep "Number of Seizures in File:" {ROOT}/chb12/chb12-summary.txt | sort | uniq -c

### 4.2 Provisional two-mode loader + chb12 classification

Handles `native_bipolar` and `rederived_cs2` only, so `_28`/`_29` raise. The
three-mode version is Section 5.

*(The vestigial `def rederive_bipolar` wrapper and the duplicate `CANONICAL` /
`norm` definitions have been removed - they now live in 0.3.)*

In [ ]:
def load_18_provisional(path):
    """Two-mode loader: native bipolar, or CS2-referential. Superseded by Section 5."""
    raw = mne.io.read_raw_edf(path, preload=True, verbose='ERROR')
    lut = {}
    for i, c in enumerate(raw.ch_names):
        lut.setdefault(norm(c), i)
    D = raw.get_data()

    if all(c in lut for c in CANONICAL):
        return D[[lut[c] for c in CANONICAL]], 'native_bipolar'

    out = np.empty((18, D.shape[1]))
    for k, pair in enumerate(CANONICAL):
        a, b = pair.split('-')
        if f'{a}-CS2' not in lut or f'{b}-CS2' not in lut:
            raise ValueError(f'{path}: cannot derive {pair}')
        out[k] = D[lut[f'{a}-CS2']] - D[lut[f'{b}-CS2']]
    return out, 'rederived_cs2'


load_18 = load_18_provisional      # replaced in Section 5

files = {}
for p in sorted(glob.glob(f'{ROOT}/chb12/*.edf')):
    names = {norm(c) for c in mne.io.read_raw_edf(p, preload=False,
                                                  verbose='ERROR').ch_names}
    files[p] = 'native_bipolar' if all(c in names for c in CANONICAL) else 'rederived'
for p, t in files.items():
    print(f'{p.split("/")[-1]:18s} {t}')

### 4.3 Quantization - RULED OUT

EDF header LSB is identical at **0.3907 uV** for both native bipolar and
CS2-referential channels. Differencing does not inherit a coarser quantization
step, so the observed power gap is not a quantization artifact.

In [ ]:
import numpy as np

def edf_lsb(path):
    with open(path, 'rb') as f:
        ns = int(f.read(256)[252:256])
        f.seek(256)
        labels = [f.read(16).decode('ascii', 'replace').strip() for _ in range(ns)]
        f.seek(256 + ns * (16 + 80 + 8))
        pmin = np.array([float(f.read(8)) for _ in range(ns)])
        pmax = np.array([float(f.read(8)) for _ in range(ns)])
        dmin = np.array([float(f.read(8)) for _ in range(ns)])
        dmax = np.array([float(f.read(8)) for _ in range(ns)])
    return labels, (pmax - pmin) / (dmax - dmin)

for p in (nat, red):
    lab, lsb = edf_lsb(p)
    print(f'\n{p.split("/")[-1]}')
    for l, s in list(zip(lab, lsb))[:4]:
        print(f'   {l:14s} LSB = {s:.4f} uV')

        #Conclusion: Not the problem

### 4.4 Power comparisons - superseded by 4.5, kept as a record

First run used the broken parser, n=1, on the window 600-900 s. `chb12_27`'s
first seizure starts at **916 s**, so that window ended 16 s before onset -
**preictal, not interictal**.

Re-run with the fixed parser and ictal +/- 120 s masking moved in-band power only
from -95.5 to -95.2 dB. **Excluding six seizures barely changed the number**, so
the elevation is not driven by ictal content. That is finding #2 in 4.5.

The Cohen's *d* here is meaningless at n=1 (std = 0 by construction). Superseded
by the spectral analysis below, which does not depend on group size.

In [ ]:
import numpy as np
from scipy import signal

nat = next(p for p, t in files.items() if t == 'native_bipolar')
red = next(p for p, t in files.items() if t == 'rederived')

for path in (nat, red):
    X, tag = load_18(path)
    p99 = np.percentile(np.abs(X), 99)
    assert 1e-6 < p99 < 1e-2, f'{tag}: {p99:.2e} V is not scalp EEG'

    f, P = signal.welch(X, fs=256, nperseg=1024, axis=-1)
    floor = P[:, (f >= 70) & (f <= 120)].mean()
    print(f'{path.split("/")[-1]:18s} {tag:15s} '
          f'p99={p99*1e6:6.1f} uV   noise floor {10*np.log10(floor):7.1f} dB')

In [ ]:
# Comparing in-band on interictal segments only

from scipy import signal

for path in (nat, red):
    X, tag = load_18(path)
    seg = X[:, 256*600 : 256*900]          # 5 min, pick a seizure-free stretch
    f, P = signal.welch(seg, fs=256, nperseg=1024, axis=-1)
    band = P[:, (f >= 1) & (f <= 40)].mean()
    print(f'{tag:15s} in-band (1-40 Hz) power {10*np.log10(band):7.1f} dB   '
          f'p99 {np.percentile(np.abs(seg),99)*1e6:6.1f} uV')

### 4.5 Spectral resolution - the analysis that settled it

Compares `chb12_27` against `chb12_24`, chosen because it is adjacent in
recording time and therefore least confounded by state drift.

**Read the output this way:**

- **PSD gap flat across frequency** -> gain/units error in the subtraction
- **PSD gap widening with frequency** -> noisier recording, arithmetic fine
- **Gap uniform across 18 channels** -> global session issue
- **Gap driven by 3-4 channels** -> bad electrode propagating into every pair

**Result: gap is ~10 dB at 3 Hz widening to ~25 dB by 30 Hz, uniform across
channels.** A scaling error shifts a log-log PSD vertically and *uniformly*.
This does not. The arithmetic is correct.

**Per-channel spread is 8.4 dB (-94.3 to -102.7), no outlier.** Left hemisphere
runs consistently hotter than right - a plausible asymmetry in a focal epilepsy
patient, not an instrumentation fault.

**Peaks: 16.25 (10.0 dB), 27.50 (4.4), 32.38 (6.7), 36.00 (3.2), 43.88 (15.4).**
32.38 is the second harmonic of 16.25. Harmonic structure implies a periodic
mechanical or electrical source - pump, motor, switching supply - not anything
neural and not an artifact of the subtraction.

### Mechanism

A bipolar amplifier rejects common-mode interference **in hardware, at the input
stage**. Software differencing of two referential channels does the same
subtraction **after** both are amplified and digitized, so interference that
affected the front end does not cancel as cleanly. Same mathematics, worse
conditions.

### Carries forward to feature extraction

43.88 Hz falls in the 0.5-40 Hz stopband and never reaches a feature. **16.25,
27.50, 32.38 and 36.00 Hz do** - they sit in beta and low gamma, and will
inflate `relpow_beta` and steepen `aperiodic_slope` on these three files.

-> `derivation` is a **covariate to check at baseline time**, not just a recorded
column.

**No global notch.** Three files out of 686; a global filter makes every file pay
the compute and the group delay. If worth doing, do it as an ablation on the
chb12 subset.

**Passband stays 0.5-40 Hz.** Excluding the in-band peaks would mean cutting to
~15 Hz and discarding beta entirely - a bad trade, since seizure discharges have
real beta content. Separately, raising the high-pass corner *increases*
delta-band group delay (35 ms @ 0.5 Hz -> 73 ms @ 1.0 -> 123 ms @ 2.0).

In [ ]:
# 4.5 - spectral resolution
import numpy as np, matplotlib.pyplot as plt
from scipy import signal

nat_p = f'{ROOT}/chb12/chb12_24.edf'   # adjacent in time, least state drift
red_p = f'{ROOT}/chb12/chb12_27.edf'

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for path, lab in [(nat_p, 'native'), (red_p, 'rederived')]:
    X, _ = load_18(path)
    m = np.ones(X.shape[1], bool)
    for s0, s1 in sz[path.split('/')[-1]].seizures:
        m[max(0, int((s0 - 120) * 256)):int((s1 + 120) * 256)] = False
    f, P = signal.welch(X[:, m], fs=256, nperseg=1024, axis=-1)
    ax[0].semilogy(f, P.mean(0), label=lab)
    ax[1].bar(np.arange(18) + (0 if lab == 'native' else 0.4),
              10 * np.log10(P[:, (f >= 1) & (f <= 40)].mean(1)), width=0.4, label=lab)
ax[0].set_xlim(0, 60); ax[0].set_xlabel('Hz'); ax[0].set_title('mean PSD'); ax[0].legend()
ax[1].set_xlabel('channel index'); ax[1].set_title('1-40 Hz power by channel'); ax[1].legend()
plt.tight_layout(); plt.show()

# per-channel breakdown: is one bad electrode driving this?
X, _ = load_18(red_p)
f, P = signal.welch(X, fs=256, nperseg=2048, axis=-1)
band = P[:, (f >= 1) & (f <= 40)].mean(1)
for i, c in enumerate(CANONICAL):
    print(f'{c:10s} {10 * np.log10(band[i]):7.1f} dB')

# narrowband peaks: harmonic structure would indicate an instrumental source
m = P.mean(0)
sel = (f >= 1) & (f <= 45)
pk, props = signal.find_peaks(10 * np.log10(m[sel]), prominence=3)
print()
for j, idx in enumerate(pk):
    print(f'{f[sel][idx]:6.2f} Hz   prominence {props["prominences"][j]:.1f} dB')

---
# 5. Canonical loader - DONE

**Acceptance passed: 683 native / 1 cs2 / 2 bare across all 686 files.**

Design points that earned their place:

- **Ordered mode check** - native, then CS2, then bare - so a recorded signal
  always beats a derived one.
- **Single exit point for the unit assertion.** A scaling error on one
  derivation mode and not another is exactly the bug that survives to the
  feature table. One check at one exit cannot be skipped by a branch added
  later.
- **`lut.setdefault`** so the first `T8-P8` wins when MNE has produced
  `T8-P8-0` / `T8-P8-1`.

> **Debugging note.** The acceptance test first returned
> `Counter({'FAILED: TypeError': 686})` - every file, including natives that had
> just worked. Cause: a docstring-only stub shadowed the working function, so
> `load_18` returned `None`. Two lessons: never write bare `except Exception` in
> a diagnostic (it converted a specific error into a tally), and define each
> symbol **once**, in Section 0. A stale `norm()` without the `01 -> O1` alias
> caused the same class of failure minutes later.

> **Cost note.** Classification only needs channel *names*, so it reads headers
> with `preload=False` (~1 min). The first attempt used `preload=True` across
> all 686 files - 45.8 GB decoded to float64, 15-30 min. Only feature
> extraction justifies a full-sample pass.

In [ ]:
def load_18(path):
    """
    Returns (data, derivation) with data shape (18, n_samples) in volts,
    channels in canonical order.

    derivation is one of:
      'native_bipolar'  canonical names present -> select, no arithmetic
      'rederived_cs2'   'A-CS2' and 'B-CS2' present -> subtract
      'rederived_bare'  bare 'A' and 'B' present -> subtract; alias 01 -> O1

    Raises rather than returning a partial or misordered array.
    Asserts 1e-6 < p99(|data|) < 1e-2 volts on every path.
    """

### 5.1 Acceptance test

```python
from collections import Counter
modes = Counter()
for p in sorted(glob.glob('/content/chbmit/chb*/*.edf')):
    try:
        X, tag = load_18(p)
        modes[tag] += 1
    except Exception as e:
        modes[f'FAILED: {type(e).__name__}'] += 1
print(modes)          # expect {'native_bipolar': 683, 'rederived_cs2': 1, 'rederived_bare': 2}
```

In [ ]:
# acceptance test - run once load_18 is written

### 5.2 chb12 group comparison at n=3 - CLOSED

```
native    n=21   -104.6 +/- 1.0 dB   range [-106.9, -102.7]
rederived n=3    -102.0 +/- 4.8 dB   [-95.2, -106.0, -104.8]
```

**`_28` (-106.0) and `_29` (-104.8) fall inside the native range.** The 4.8 dB
spread is one file, not a group property.

Re-derivation introduces **no detectable power shift**. The elevation is
specific to the `chb12_27` session, not to the method.

Amplitudes confirm the one assumption the header could not verify - an unstated
common reference on the bare-label files:

| file | mode | p99 |
|---|---|---|
| `chb01_03` | native_bipolar | 144.8 uV |
| `chb12_27` | rederived_cs2 | 414.6 uV |
| `chb12_28` | rederived_bare | 108.2 uV |
| `chb12_29` | rederived_bare | 158.6 uV |

All in normal scalp range. Had the assumption been wrong, the unit assertion at
`load_18`'s single exit point would have fired.

-> `derivation` alone is too coarse a covariate. Flag by **record** instead.

*(`chb12_29` is 927,744 samples vs 921,600 - 3,624 s, matching its summary end
time of 19:07:43. Legitimately 24 s longer.)*

**All 13 seizures recovered.**

In [ ]:
# n=3 comparison (requires `files` rebuilt with the 3-mode classifier)
rows = [r for p, t in files.items() if (r := clean_power(p, t))]
nat = [r[0] for r in rows if r[2] == 'native_bipolar']
red = [r[0] for r in rows if r[2] != 'native_bipolar']
print(f'native    n={len(nat)}  {np.mean(nat):.1f} +/- {np.std(nat):.1f} dB  '
      f'range [{min(nat):.1f}, {max(nat):.1f}]')
print(f'rederived n={len(red)}  {np.mean(red):.1f} +/- {np.std(red):.1f} dB  '
      f'{[f"{x:.1f}" for x in red]}')

# Conclusion: _27 carries session-specific interference, _28/_29 do not.
# NOISY_RECORDS is defined operationally in section 7.

---
# 6. Causal filtering - DONE

2nd-order Butterworth bandpass, 0.5-40 Hz, forward-only `sosfilt` with `zi`
carried across blocks.

**On "2nd order":** for a bandpass design scipy's `order` produces a filter of
realised order `2*order`. `order=2` here is a 4th-order bandpass = 2 SOS
sections.

**`zi` shape is `(n_sections, n_channels, 2)`.** `sosfilt_zi` returns
`(n_sections, 2)` and must be broadcast across channels - the usual stumbling
block.

**Priming matters more than expected.** On a DC-offset signal the first 64
samples cold-start at 5.07e-05 against a 1.49e-06 steady state - the offset
passes straight through before the highpass settles. Priming `zi` to the first
sample gives 1.88e-06, essentially steady state immediately. **27x less startup
garbage**, and it matters because windowing starts at sample zero.

**`reset()` is per-record, never per-session.** Filter state must not cross
recording boundaries - different sessions, sometimes hours apart. Carrying it
smears the tail of one record into the head of the next, invisibly.

In [ ]:
class CausalBandpass:
    """Forward-only Butterworth bandpass with persistent per-channel state."""

    def __init__(self, sfreq=256.0, l_freq=0.5, h_freq=40.0, order=2, n_channels=18):
        nyq = sfreq / 2.0
        if not 0 < l_freq < h_freq < nyq:
            raise ValueError(f'bad band {l_freq}-{h_freq} Hz for sfreq={sfreq}')
        self.sfreq, self.n_channels = float(sfreq), int(n_channels)
        self.l_freq, self.h_freq, self.order = l_freq, h_freq, order
        self.sos = signal.butter(order, [l_freq/nyq, h_freq/nyq],
                                 btype='band', output='sos')
        self._zi_proto = signal.sosfilt_zi(self.sos)      # (n_sections, 2)
        self.reset()

    def reset(self, primer=None):
        """Clear state. `primer` (n_channels,) initialises to steady state for
        that DC level, avoiding a startup transient."""
        zi = np.repeat(self._zi_proto[:, None, :], self.n_channels, axis=1)
        self._zi = zi * (np.asarray(primer, float)[None, :, None]
                         if primer is not None else 0.0)

    def process(self, block):
        block = np.asarray(block, dtype=np.float64)
        if block.ndim != 2 or block.shape[0] != self.n_channels:
            raise ValueError(f'expected ({self.n_channels}, n), got {block.shape}')
        out, self._zi = signal.sosfilt(self.sos, block, axis=-1, zi=self._zi)
        return out

    def group_delay_ms(self, freqs):
        b, a = signal.sos2tf(self.sos)
        _, gd = signal.group_delay((b, a), w=np.asarray(freqs, float), fs=self.sfreq)
        return gd / self.sfreq * 1e3

### 6.1 Equivalence test - the load-bearing artifact

Filtering one pass over a whole record must be **identical** to streaming it in
4-sample packets with state carried. That result is what licenses offline
whole-record filtering while still claiming stream-equivalence.

**Result: `0.000e+00` - bit-identical, not merely within tolerance.**

The negative controls are what make this a real test rather than a tautology.
Both failure modes are the same order of magnitude as the signal itself
(2.6e-04), and both are silent - you would get plausible features and an
undeployable model:

| comparison | max abs diff |
|---|---|
| streamed **with** state carry | **0.000e+00** |
| streamed **without** state carry | 2.091e-04 |
| `filtfilt` (non-causal) | 4.445e-04 |

In [ ]:
def test_streaming_equivalence(packet=4, n=4096, n_channels=18, seed=0):
    rng = np.random.default_rng(seed)
    x = rng.standard_normal((n_channels, n)) * 1e-4

    f1 = CausalBandpass(n_channels=n_channels); whole = f1.process(x)
    f2 = CausalBandpass(n_channels=n_channels)
    streamed = np.concatenate([f2.process(x[:, i:i+packet])
                               for i in range(0, n, packet)], axis=-1)

    diff = np.abs(whole - streamed).max()
    print(f'streaming equivalence: max abs diff {diff:.3e}  '
          f'(signal scale {np.abs(whole).max():.2e})')
    assert diff < 1e-12, 'state is not being carried across blocks'

    naive = np.concatenate([signal.sosfilt(f1.sos, x[:, i:i+packet], axis=-1)
                            for i in range(0, n, packet)], axis=-1)
    ff = signal.sosfiltfilt(f1.sos, x, axis=-1)
    print(f'  no state carry: {np.abs(whole-naive).max():.3e}  <- what this catches')
    print(f'  filtfilt:       {np.abs(whole-ff).max():.3e}  <- non-causal, never use')
    return diff

test_streaming_equivalence()

bp = CausalBandpass()
for f, d in zip([2,6,10,20,35], bp.group_delay_ms(np.array([2.,6.,10.,20.,35.]))):
    print(f'  {f:2d} Hz  {d:5.1f} ms' + ('   <- exceeds 15.6 ms packet budget'
                                          if d > 15.625 else ''))

---
# 7. Windowing and labelling - DONE

2 s windows, 1 s stride, 60 s guard band.

**Parameterized by duration, not sample count** (D21). Frequency resolution
depends on window *duration* - a 2 s window gives 0.5 Hz bins at both 256 and
128 Hz. Hardcoding `512` is what would make the sampling-rate decision expensive.

**Ictal is applied before the guard band, so ictal wins.** A window straddling
the guard of one seizure and the onset of the next is correctly ictal. Verified:
with seizures at 50-55 s and 70-75 s and a 20 s guard, `t=69` (window [69,71])
stays ictal.

**`sliding_window_view` returns a view, not a copy** - confirmed by
`shares_memory`. Saves a 2x blowup at 50% overlap, but the windows alias the
filtered array, so do not mutate either in place.

**Verified edge cases:** guard bands merge between close seizures; a seizure at
`t=0` does not crash on a negative guard; event ids survive for
leave-one-seizure-out.

In [ ]:
LAB_ICTAL, LAB_INTER, LAB_DROP = 1, 0, -1
SUBJECT_ALIASES = {'chb21': 'chb01'}      # same patient, 1.5 yr apart
NOISY_RECORDS   = {'chb12_27.edf'}        # session-specific narrowband interference
WINDOW_SEC, STRIDE_SEC, GUARD_SEC = 2.0, 1.0, 60.0


def subject_group_of(record):
    """CV grouping key. Folds chb21->chb01 and chb17a/b/c->chb17."""
    s = record.split('_')[0]
    s = SUBJECT_ALIASES.get(s, s)
    return 'chb17' if s.startswith('chb17') else s


def label_windows(starts, window_sec, seizures, guard_sec):
    """Returns (labels, event_idx). Ictal takes precedence over the guard band."""
    ends = starts + window_sec
    lab = np.full(starts.shape, LAB_INTER, dtype=np.int8)
    ev  = np.full(starts.shape, -1, dtype=np.int16)
    for k, (s0, s1) in enumerate(seizures):            # ictal first
        ov = (starts < s1) & (ends > s0)
        lab[ov] = LAB_ICTAL; ev[ov] = k
    for s0, s1 in seizures:                            # then guard; ictal wins
        near = (starts < s1 + guard_sec) & (ends > s0 - guard_sec)
        lab[near & (lab != LAB_ICTAL)] = LAB_DROP
    return lab, ev


def window_record(path, summary, sfreq=256.0, window_sec=WINDOW_SEC,
                  stride_sec=STRIDE_SEC, guard_sec=GUARD_SEC):
    """Load, filter causally, window, and label one record."""
    record = os.path.basename(path)

    if summary is None:
        if not summarized:
            raise RuntimeError('`summarized` is empty - rebuild it (cell 0.6)')
        if record not in KNOWN_UNANNOTATED:
            raise ValueError(f'{record}: no summary entry and not a known '
                             f'unannotated record')
        return None                          # D7: unannotated != seizure-free

    X, derivation = load_18(path)

    bp = CausalBandpass(sfreq=sfreq, n_channels=18)
    bp.reset(primer=X[:, 0])                 # per-record; state never crosses records
    X = bp.process(X)

    w_size = int(round(window_sec * sfreq))   # D21: duration, not sample count
    w_stride = int(round(stride_sec * sfreq))
    if X.shape[1] < w_size:
        return None
    W = sliding_window_view(X, w_size, axis=-1)[:, ::w_stride, :].transpose(1, 0, 2)

    starts = np.arange(W.shape[0], dtype=np.float64) * stride_sec
    lab, ev = label_windows(starts, window_sec, summary.seizures, guard_sec)

    keep = lab != LAB_DROP
    if not keep.any():
        return None

    meta = {
        'y': lab[keep], 't_start': starts[keep], 'record': record,
        'subject': record.split('_')[0],
        'subject_group': subject_group_of(record),
        'seizure_event': [f'{record}#{e}' if e >= 0 else '' for e in ev[keep]],
        'derivation': derivation,
        'noisy': record in NOISY_RECORDS,
    }
    return W[keep], lab[keep], meta

---
# 8. Feature extraction - DONE, 26/26 shards

291 features: 18 channels x 16 + 3 artifact guards.

Everything constant is precomputed at construction: the Hann taper, the
frequency masks, and the log-log design-matrix pseudo-inverse. The hot path
allocates only its output and never loops over channels in Python.

### Three findings from building this

**1. The epsilon bug - caught only by differential testing.** Shape was right,
finiteness was right, nothing was NaN. But EEG in **volts** has variance ~3e-9,
so a fixed `_EPS = 1e-12` is a ~0.03% perturbation on a typical channel and far
worse on quiet ones. Hjorth mobility disagreed with a naive implementation by
**0.008** - a 1% error on a scale-invariant feature.

Fix: **work in microvolts.** `_EPS` becomes 3e-14% of variance, and
`hjorth_activity` reads as ~3.5 instead of -8.5. After the fix both differential
tests land at ~1.4e-08 and ~2.6e-08, which is the **float32 storage floor**
(`np.spacing(float32(1.0))` ~ 6e-8), not residual error.

> Epsilon guards must be scaled to the data. A constant that is inert at unit
> scale can be a 1% distortion twelve orders of magnitude down.

**2. DWT bands do not fit the passband.** At 256 Hz, a 4-level db4 decomposition
gives cD1 = 64-128 Hz (**entirely stopband**) and cD2 = 32-64 Hz (mostly
stopband). Two of five DWT features per channel - 36 of 291 - quantify filter
rolloff noise. Combined with the op count (15,872 MACs/channel vs 5,377 for a
*shared* rFFT), the DWT is likely to lose its ablation. Note that at 128 Hz the
bands realign (cD1 = 32-64, cD2 = 16-32) - another point for decimation.

**3. `wavedec` returns details in reverse frequency order** -
`[cA_n, cD_n, ..., cD1]`. Naming them cD1..cDn in list order silently mislabels
every band and nothing downstream would catch it.

### Verified

- closed-form aperiodic slope vs `np.polyfit`: **8.77e-15**, and **300x faster**
  (0.002 ms vs 0.623 ms). The pinv is `(2, 79)` and constant - this is D10.
- Hjorth mobility and line length differential tests at the float32 floor.

### 8.2 Corpus verification - the gate before modelling

Reads **columns only**. `pd.read_parquet` on the whole features directory loads
~4 GB before pandas overhead and has crashed the runtime once.

The event count is the check that catches silent seizure loss - the failure mode
that hit twice (a whitespace bug in the parser, then an empty `summarized` dict).

**Result: 198/198 events, 23 groups.**

In [ ]:
rows = []
for f in sorted(glob.glob(f'{FEATURES}/*.parquet')):
    d = pd.read_parquet(f, columns=['y','subject_group','seizure_event'])
    rows.append({'shard': os.path.basename(f)[:-8],
                 'group': str(d.subject_group.iloc[0]),
                 'windows': len(d), 'ictal': int((d.y==1).sum()),
                 'events': d.loc[d.y==1,'seizure_event'].nunique()})
    del d
m = pd.DataFrame(rows)
print(m.to_string(index=False))
print(f'\nwindows {m.windows.sum():,} | ictal {m.ictal.sum():,} '
      f'({m.ictal.sum()/m.windows.sum()*100:.3f}%)')
print(f'events {m.events.sum()} (expect 198) | groups {m.group.nunique()} (expect 23)')

assert m.events.sum() == 198, f'SEIZURE LOSS: {m.events.sum()}'
assert m.group.nunique() == 23, f'grouping wrong: {m.group.nunique()}'
print('\nPASS')

### 8.3 Corpus shape - three facts that constrain modelling

**Positives are extremely concentrated.** chb12 (40 events), chb15 (20),
chb24 (16), chb13 (12) hold 88 of 198 - **44% from four subjects** - while
chb02/07/11/19/22 have 3 each.

**Ictal windows per event span 29x.** chb11 averages 270 windows/seizure;
chb16 averages 9.4. Real seizure-duration variation, and it caps achievable
sensitivity on short-seizure subjects regardless of model quality.

**chb04 is 16% of the table from 4 of 198 seizures.** Its records are ~4 hours
(median `t_start` max 14,398 s), not one. Pooled, it contributes 16% of negatives
and 2% of positives.

-> **These are why D26 exists: report per-fold, never the mean alone.**

In [ ]:
d = pd.read_parquet(f'{FEATURES}/chb04.parquet',
                    columns=['record','t_start'])
print('chb04 record durations (s):')
print(d.groupby('record', observed=True).t_start.max().describe())
del d

m['win_per_event'] = (m.ictal / m.events.replace(0, np.nan)).round(1)
print('\nictal windows per seizure event:')
print(m[['shard','events','ictal','win_per_event']]
      .sort_values('win_per_event').to_string(index=False))

---
# 9. Evaluation harness

Written **before** the models, because it is what determines whether a model is
any good.

## Do not report accuracy

Ictal windows are 0.339% of the corpus. A null model scores **99.66%**.
Accuracy appears nowhere in this project.

## Read `auprc_lift`, not `auprc`

At this base rate a useless model scores AUPRC ~0.0034. An AUPRC of 0.05 is a
**15x lift** and genuinely good, while looking terrible in isolation. AUPRC
without its baseline is uninterpretable.

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score

def window_metrics(y_true, scores):
    """Threshold-free ranking metrics. AUPRC is meaningless without its baseline."""
    y_true = np.asarray(y_true)
    pos_rate = float(y_true.mean())
    out = {'positive_rate': pos_rate, 'auprc_baseline': pos_rate}
    if len(np.unique(y_true)) < 2:
        return {**out, 'auroc': np.nan, 'auprc': np.nan, 'auprc_lift': np.nan}
    out['auroc'] = float(roc_auc_score(y_true, scores))
    out['auprc'] = float(average_precision_score(y_true, scores))
    out['auprc_lift'] = out['auprc'] / pos_rate
    return out

### 9.1 Verification - known-answer cases at realistic imbalance

The `random` case is the important one: **lift ~1.0x** confirms that at a 0.34%
base rate a useless model scores AUPRC ~0.0034.

The single-class check is not hypothetical - **chb17c has zero ictal windows**,
so a fold with no positives must return `nan` rather than crash the CV loop.

In [ ]:
def verify_window_metrics():
    rng = np.random.default_rng(0)
    n = 100_000
    y = (rng.random(n) < 0.0034).astype(int)          # CHB-MIT-like base rate
    print(f'n={n:,}  positives={y.sum()}  rate={y.mean():.5f}\n')

    cases = {'perfect':  y + rng.random(n) * 1e-9,
             'random':   rng.random(n),
             'inverted': -(y + rng.random(n) * 1e-9),
             'weak':     y * 0.5 + rng.random(n)}
    res = {}
    for name, s in cases.items():
        m = window_metrics(y, s); res[name] = m
        print(f'{name:9s} AUROC {m["auroc"]:.3f}  AUPRC {m["auprc"]:.4f}  '
              f'lift {m["auprc_lift"]:7.1f}x')

    assert res['perfect']['auprc'] > 0.99
    assert 0.7 < res['random']['auprc_lift'] < 1.4, 'random must sit at baseline'
    assert res['inverted']['auprc_lift'] < 1.0
    assert res['weak']['auroc'] > res['random']['auroc']

    m = window_metrics(np.zeros(100, int), rng.random(100))
    assert np.isnan(m['auprc']) and np.isnan(m['auroc'])
    print('\nsingle-class fold returns nan without crashing')
    print('PASS')

verify_window_metrics()

### 9.2 Per-fold reporting - D26

**44% of positives come from four subjects. Five subjects have 3 events each.**

A mean across 23 LOSO folds averages over a distribution that is nowhere near
uniform, and it hides both the subjects where the model fails completely and the
ones carrying the result.

Every results table prints the **per-fold rows**, the **spread**, and the
**worst fold by name**. Never the mean alone.

In [ ]:
def report_folds(per_fold, metrics=('auprc','auprc_lift','auroc')):
    """Print per-fold results, then the summary. Never the mean alone (D26)."""
    df = pd.DataFrame(per_fold)
    print(df.to_string(index=False))
    print()
    for mname in metrics:
        if mname not in df: continue
        v = df[mname].dropna()
        if not len(v): continue
        worst = df.loc[v.idxmin()]
        print(f'{mname:12s} mean {v.mean():7.4f}  median {v.median():7.4f}  '
              f'min {v.min():7.4f}  max {v.max():7.4f}  '
              f'(worst fold: {worst.get("held_out","?")})')
    return df

### 9.3 Still to write

**Event-level sensitivity at a fixed false-alarm rate per hour**, and
**detection latency** from annotated onset, under a k-of-n persistence rule.

> **The subsampling correction must be built in from the start.** If interictal
> windows are kept at rate *r*, each surviving negative represents 1/*r* seconds
> of recording, so the FA/h denominator must be divided by *r*. Retrofitting
> this is how a paper ends up reporting a 10x optimistic false-alarm rate.

### 9.4 Event-level metrics and quantile threshold selection

These are the harness the classical baselines, the calibration curve and
the TCN are all scored through. `event_metrics` is what makes the
comparison across model classes meaningful.

`select_quantile`, which returns a quantile rather than an absolute
threshold, is defined in section 12.5 alongside the evidence that forced
that choice.

In [ ]:
def _persistence(alarm, k, n):
    """k-of-n smoothing: fire only if k of the last n windows exceeded threshold."""
    if k <= 1 and n <= 1:
        return alarm
    c = np.convolve(alarm.astype(int), np.ones(n, int), mode='full')[:len(alarm)]
    return c >= k


def event_metrics(meta, scores, threshold, window_sec=2.0, stride_sec=1.0,
                  k_of_n=(3, 5), refractory_sec=30.0, tolerance_sec=30.0,
                  subsample_rate=1.0):
    """Event-level sensitivity, false alarms per hour, and detection latency.

    `meta` needs columns: record, t_start, y, seizure_event.

    subsample_rate: if interictal windows were kept at rate r, each survivor
    represents 1/r seconds of recording, so the FA/h denominator divides by r
    (D30). Omitting this reports a 1/r-times optimistic false-alarm rate.

    CAVEAT (D31): refractory merging collapses continuous alarming into a single
    "event", so an always-on model scores a deceptively low FA/h. A random model
    once posted sensitivity 1.00 at 2.02 FA/h with duty 0.50. Always read
    alarm_duty alongside it.
    """
    meta = meta.reset_index(drop=True).copy()
    meta['score'] = np.asarray(scores)
    k, n = k_of_n
    detected, latencies = [], []
    n_fa = n_inter = n_alarm_inter = 0

    for _, grp in meta.groupby('record', observed=True, sort=False):
        grp = grp.sort_values('t_start')
        t, y = grp.t_start.to_numpy(), grp.y.to_numpy()
        ev = grp.seizure_event.to_numpy()
        alarm = _persistence(grp.score.to_numpy() >= threshold, k, n)
        t_dec = t + window_sec              # a decision is available at window END

        for eid in pd.unique(ev[y == 1]):
            m = ev == eid
            onset = float(t[m].min())
            offset = float(t[m].max()) + window_sec
            hit = alarm & (t_dec >= onset) & (t_dec <= offset + tolerance_sec)
            detected.append(bool(hit.any()))
            if hit.any():
                latencies.append(float(t_dec[hit].min() - onset))

        inter = y == 0
        n_inter += int(inter.sum())
        fa = alarm & inter
        n_alarm_inter += int(fa.sum())
        if fa.any():
            ft = t_dec[fa]
            n_fa += 1 + int((np.diff(ft) > refractory_sec).sum())

    inter_hours = n_inter * stride_sec / 3600.0 / subsample_rate
    return {
        'threshold': float(threshold),
        'n_events': len(detected),
        'sensitivity_event': float(np.mean(detected)) if detected else np.nan,
        'false_alarms': n_fa,
        'interictal_hours': inter_hours,
        'fa_per_hour': n_fa / inter_hours if inter_hours > 0 else np.nan,
        'alarm_duty': n_alarm_inter / n_inter if n_inter else np.nan,
        'latency_median_s': float(np.median(latencies)) if latencies else np.nan,
        'latency_p90_s': float(np.percentile(latencies, 90)) if latencies else np.nan,
    }

The superseded absolute-threshold rule is kept because figure 3 shows
it failing: max test duty 0.0903 against 0.0071 for quantile transfer.

In [ ]:
def select_threshold(scores, meta, target_fa=2.0, max_duty=0.01,
                     n_grid=40, **kw):
    """SUPERSEDED by select_quantile (D44). Kept only to generate figure 3,
    which shows this rule failing: max test duty 0.0903 against 0.0071."""
    cands = np.unique(np.quantile(scores, np.linspace(0.90, 0.9995, n_grid)))
    best = cands[-1]
    for thr in cands:
        m = event_metrics(meta, scores, float(thr), **kw)
        if np.isnan(m['alarm_duty']) or m['alarm_duty'] > max_duty:
            continue
        if not np.isnan(m['fa_per_hour']) and m['fa_per_hour'] > target_fa:
            continue
        best = float(thr); break
    return best

---
# 10. LOSO baselines

Training folds subsampled to 10% of interictal windows (all positives kept).
**Test folds at full density** - k-of-n persistence and detection latency both
require contiguous windows, so a subsampled test fold makes both meaningless
(D29).

The scaler lives **inside** the sklearn Pipeline. Fitting it on the full table
before splitting is the classic silent leak.

### 10.0 Model constructors

The scaler lives inside the Pipeline, so it is fit on training folds
only. Fitting it on the full table before splitting is the classic
silent leak.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

In [ ]:
def load_subjects(groups, subsample=1.0, seed=0, features=f'{FEATURES}'):
    """Keeps ALL positives; samples negatives at `subsample`.
    Pass subsample=1.0 for TEST folds."""
    rng = np.random.default_rng(seed)
    frames = []
    for f in sorted(glob.glob(f'{features}/*.parquet')):
        d = pd.read_parquet(f)
        d = d[d.subject_group.isin(groups)]
        if not len(d):
            del d; continue
        if subsample < 1.0:
            keep = (d.y == 1).to_numpy() | (rng.random(len(d)) < subsample)
            d = d[keep]
        frames.append(d); del d
    out = pd.concat(frames, ignore_index=True)
    del frames; gc.collect()
    return out

def split_xy(df):
    feat = [c for c in df.columns if c not in META_COLS]
    return (df[feat].to_numpy(dtype=np.float32),
            df.y.to_numpy(dtype=np.int8), df[list(META_COLS)].copy())

def make_model(name, seed=0):
    if name == 'logreg':
        return Pipeline([('scale', StandardScaler()),
                         ('clf', LogisticRegression(max_iter=2000,
                                                    class_weight='balanced', C=1.0))])
    if name == 'svm_linear':
        return Pipeline([('scale', StandardScaler()),
                         ('clf', LinearSVC(C=0.1, class_weight='balanced',
                                           dual='auto', max_iter=5000))])
    if name == 'rf':
        return Pipeline([('clf', RandomForestClassifier(
            n_estimators=100, min_samples_leaf=5, max_features='sqrt',
            class_weight='balanced_subsample', n_jobs=-1, random_state=seed))])
    raise ValueError(name)

def scores_of(model, X):
    if hasattr(model[-1], 'predict_proba'):
        return model.predict_proba(X)[:, 1]
    return model.decision_function(X)

### 10.1 The loop

Note the **incremental save inside the loop** (D33). Hours of compute should not
depend on one variable assignment surviving a disconnect.

In [ ]:
def run_loso(models=('logreg',), subsample=0.1, seed=0,
             features=f'{FEATURES}', groups=None,
             save_partial=f'{OUT_DIR}/loso_folds_partial.csv'):
    if groups is None:
        groups = sorted({str(g) for f in glob.glob(f'{features}/*.parquet')
                         for g in pd.read_parquet(f, columns=['subject_group'])
                         .subject_group.unique()})
    print(f'{len(groups)} groups: {groups}\n')

    rows = []
    for held in groups:
        te = load_subjects([held], subsample=1.0, features=features)
        if (te.y == 1).sum() == 0:
            print(f'{held}: no seizures held out, skipping'); del te; continue
        Xte, yte, mte = split_xy(te); del te

        tr = load_subjects([g for g in groups if g != held],
                           subsample=subsample, seed=seed, features=features)
        Xtr, ytr, _ = split_xy(tr); del tr; gc.collect()

        for name in models:
            t0 = time.perf_counter()
            model = make_model(name, seed); model.fit(Xtr, ytr)
            s = scores_of(model, Xte)
            rec = {'model': name, 'held_out': held, 'n_train': len(ytr),
                   'n_test': len(yte),
                   'test_events': mte.loc[yte == 1, 'seizure_event'].nunique()}
            rec.update(window_metrics(yte, s))
            rec.update(event_metrics(mte, s, np.quantile(s, 0.995), subsample_rate=1.0))
            rec['fit_s'] = time.perf_counter() - t0
            rows.append(rec)
            if save_partial:
                pd.DataFrame(rows).to_csv(save_partial, index=False)
            print(f'  {name:11s} {held:7s} AUPRC {rec["auprc"]:.4f} '
                  f'(lift {rec["auprc_lift"]:5.1f}x)  sens {rec["sensitivity_event"]:.2f} '
                  f'@ {rec["fa_per_hour"]:5.2f} FA/h  duty {rec["alarm_duty"]:.3f}  '
                  f'lat {rec["latency_median_s"]:.1f}s  [{rec["fit_s"]:.0f}s]')

        del Xtr, ytr, Xte, yte, mte; gc.collect()
    return pd.DataFrame(rows)

per_fold = run_loso(models=('logreg','svm_linear','rf'), subsample=0.1)
per_fold.to_csv(f'{OUT_DIR}/loso_folds.csv', index=False)
per_fold.to_csv(f'{ARCHIVE}/loso_folds.csv', index=False)

### 10.2 Results - logistic regression, 23/23 folds

```
median AUPRC lift   28.6x        (mean 52.6x -- SKEWED, report the median)
folds at chance     6/23         (lift < 5x, sensitivity 0.00)
folds sens >= 0.8   10/23
alarm duty          0.001-0.004  every fold -- not always-on
median latency      ~22 s        against a 4 s floor
```

| held out | lift | sens | FA/h | lat |
|---|---|---|---|---|
| chb09 | 262x | 1.00 | 0.56 | 10 s |
| chb10 | 155x | 1.00 | 0.89 | 13 s |
| chb19 | 133x | 1.00 | 0.91 | 31 s |
| chb22 | 133x | 1.00 | 0.94 | 24 s |
| chb14 | 2.3x | 0.00 | 1.75 | - |
| chb06 | 2.8x | 0.00 | 0.77 | - |
| chb15 | 1.7x | 0.05 | 0.88 | 37 s |
| chb12 | 1.3x | 0.00 | 1.14 | - |
| chb13 | 1.1x | 0.00 | 1.23 | - |

Linear SVM tracks logreg almost exactly (chb01: 34.2x vs 35.7x, identical sens
and latency) - both fit a linear boundary on the same standardized features.
If that holds across folds they are redundant and one can be dropped, halving
iteration time. RF is the only nonlinear model in the set.

### 10.3 The finding: more seizures predicts *worse* detection

An initial reading of the first 14 folds suggested seizure **duration** drove
performance. **That hypothesis does not survive all 23** - chb15 has 101 windows
per event and fails at 1.7x; chb24 has 33 and succeeds at 35.8x with sensitivity
1.00. Two clean counterexamples.

The stronger relationship is with **event count, and it is negative**:

| predictor | vs lift | vs sens |
|---|---|---|
| windows per event (duration) | rho +0.427, p 0.042 | +0.504, p 0.014 |
| **number of events** | **rho -0.638, p 0.001** | **-0.563, p 0.005** |

- **<=5 events (n=10):** median lift 56.8x, median sens 0.90
- **>=10 events (n=6):** median lift 2.2x, median sens 0.00

Every subject with >=10 annotated seizures fails, except chb24.

**Two candidate mechanisms:**

*Annotation depth* - a patient with 40 marked events may have many brief
subclinical discharges a careful reviewer caught, while a patient with 3 has
three unambiguous clinical seizures. If so, the metric partly reflects reviewer
conservatism rather than detectability.

*Severity* - frequent-seizure patients may have more diffuse or variable
signatures that resist a shared cross-subject template.

Unresolved. Distinguishing them needs seizure durations read from the
annotations, not inferred from window counts.

In [ ]:
# read durations directly from the annotations
rows = [{'subject_group': subject_group_of(name), 'dur': s1 - s0}
        for name, rec in summarized.items() for s0, s1 in rec.seizures]
sz = pd.DataFrame(rows)
print(sz.groupby('subject_group').dur.agg(['count','median','min','max','std'])
        .sort_values('count', ascending=False).to_string())

### 10.4 Two caveats to fix before any write-up

**1. The threshold is selected on the test fold** (`np.quantile(s, 0.995)`).
That effectively pins the alarm rate near 1 FA/h on every fold, which makes folds
comparable but makes the sensitivity figures mildly optimistic. Replace with an
inner-validation threshold, or report the full operating curve.

**2. The best folds have the least reliable sensitivity estimates.** chb22 at
133x is 3/3 events; chb09 at 262x is 4/4. Sensitivity from three events has
enormous variance. AUPRC lift is window-level and better estimated, but the
sensitivity column needs confidence intervals.

### Latency is not the persistence rule

Floor is `window_sec + (k-1)*stride_sec` = **4 s** at 2 s / 3-of-5. Observed
medians are 10-36 s, so the smoothing rule is **not binding** - the score ramps
slowly through the seizure.

-> **The latency problem and the short-seizure problem are the same problem.**
The fix is faster features or a lower operating threshold, not a shorter
persistence window.

### 10.5 Model selection - RF as the sole baseline (D34)

Three models on chb01:

| model | lift | sens | FA/h | latency | fit |
|---|---|---|---|---|---|
| logistic regression | 35.7x | 0.64 | 1.15 | 25.0 s | 106 s |
| linear SVM | 34.2x | 0.64 | 1.11 | 25.0 s | 125 s |
| **random forest (100 trees)** | **156.4x** | 0.64 | **0.84** | **12.0 s** | 1046 s |

**The linear models are redundant** - 35.7x vs 34.2x, identical sens and latency.
Both fit a linear boundary on the same standardized features with balanced class
weights. Running both costs iteration time and answers nothing.

**RF is 4.4x better AUPRC with lower FA/h *and* half the latency.** Nonlinearity
improves both headline problems at once. Cost: 10x training time, ~6.7 h for a
full sweep.

-> RF is the sole baseline. The linear numbers already collected are reported as
a one-line comparison.

### RF fits the latency budget but not the memory budget

100 trees measured 6.7 ms p50 / 8.6 ms p99 - inside the 15.625 ms packet budget.
(300 trees was 21.7 ms and does not fit.)

| model | footprint |
|---|---|
| logistic regression | ~1.2 KB (291 coefficients) |
| random forest, 100 trees | ~340 KB (~28,176 nodes x ~12 B) |

**280x difference.** 340 KB exceeds typical MCU SRAM though it fits flash as a
read-only structure. The accuracy-vs-deployability trade is now concrete.

### 10.6 The gate before the TCN (D35)

RF's 156x on chb01 is encouraging - but chb01 was **already a working fold** for
logistic regression. The question that matters:

> **Does RF rescue the four folds at chance - chb06, chb12, chb13, chb15?**

- **If yes:** the failure was model capacity. A TCN is the natural escalation.
- **If no:** the failure is features, labels, or cross-subject calibration - and
  a TCN will fail the same way at 50x the compute.

Two hours of compute to answer a question that determines the next month.
**Run this before the full sweep.**

In [ ]:
# ~2 h. Answer this before committing 6.7 h to the full RF sweep.
rf_hard = run_loso(models=('rf',), subsample=0.1,
                   groups=['chb06','chb12','chb13','chb15'],
                   save_partial=f'{OUT_DIR}/rf_hard_partial.csv')
rf_hard.to_csv(f'{OUT_DIR}/rf_hard_folds.csv', index=False)
rf_hard.to_csv(f'{ARCHIVE}/rf_hard_folds.csv', index=False)

# logreg lifts on these folds were: chb06 2.8x, chb12 1.3x, chb13 1.1x, chb15 1.7x
print(rf_hard[['held_out','auprc_lift','sensitivity_event',
               'fa_per_hour','latency_median_s']].to_string(index=False))

### 10.7 Full RF sweep

Only after 10.6. ~6.7 h across 23 folds, but the incremental save means a
disconnect costs nothing - restart and it resumes from the partial CSV.

If 6.7 h is too long: `n_estimators=50` roughly halves it (the tree-count trade
curve is already measured), or `subsample=0.05` halves the training set.

In [ ]:
per_fold_rf = run_loso(models=('rf',), subsample=0.1,
                       save_partial=f'{OUT_DIR}/rf_partial.csv')
per_fold_rf.to_csv(f'{OUT_DIR}/loso_folds_rf.csv', index=False)
per_fold_rf.to_csv(f'{ARCHIVE}/loso_folds_rf.csv', index=False)
report_folds(per_fold_rf)

---
# 11. Why cross-subject transfer fails - four experiments

Six of 23 LOSO folds sit at chance. Hypotheses tested in increasing cost order;
all but the last eliminated.

### 11.1 Is it model capacity? - NO

| fold | logreg | RF (100 trees) |
|---|---|---|
| chb06 | 2.8x | 1.7x |
| chb12 | 1.3x | 2.4x |
| chb13 | 1.1x | **0.8x** |
| chb15 | 1.7x | **0.9x** |

RF gave **156x on chb01** with the same features and training data. A model with
the capacity to reach 156x elsewhere is not failing here for expressiveness.
chb13 and chb15 fall *below* chance - ranking slightly anti-correlated with truth.

This also rules out **decision-boundary geometry**: RF (axis-aligned partitions)
and logreg (a global hyperplane) have opposite inductive biases and fail
identically. A third model class would not have been informative.

In [ ]:
rf_hard = run_loso(models=('rf',), subsample=0.1,
                   groups=['chb06','chb12','chb13','chb15'],
                   save_partial=f'{OUT_DIR}/rf_hard_partial.csv')

### 11.2 Do the features work at all? - YES

Patient-specific RF on chb15, leave-one-**record**-out (whole records held out,
so no window overlap across the split):

| held out | lift |
|---|---|
| chb15_06 | 26.2x |
| chb15_10 | **107.3x** |
| chb15_15 | 22.0x |
| chb15_17 | 77.7x |
| chb15_20 | 56.2x |

**Median ~56x against a cross-subject 0.9x.** Rules out three things at once:
the features capture chb15's seizures, the labels are correct (you cannot get
107x against wrong labels), and it is not capacity.

> **This gives a CEILING.** Every generalization technique from here is measured
> as *what fraction of the within-subject ceiling it recovers* (D40) - a better
> target than raw AUPRC, and one that normalizes across subjects of differing
> difficulty.

In [ ]:
d15 = load_subjects(['chb15'], subsample=1.0)
for held in sorted(d15.loc[d15.y==1,'record'].unique())[:5]:
    te, tr = d15[d15.record==held], d15[d15.record!=held]
    if (te.y==1).sum()==0 or (tr.y==1).sum()==0: continue
    Xtr,ytr,_ = split_xy(tr); Xte,yte,mte = split_xy(te)
    m = make_model('rf'); m.fit(Xtr,ytr); s = scores_of(m,Xte)
    print(f'{held}: lift {window_metrics(yte,s)["auprc_lift"]:6.1f}x  (LOSO gave 0.9x)')

### 11.3 Is it scale/offset mismatch? - NO

Within-**record** z-scoring, median-centred so ictal windows do not shift the
baseline:

| fold | unnormalized | normalized |
|---|---|---|
| chb06 | 2.8x | 1.3x |
| chb12 | 1.3x | 1.3x |
| chb13 | 1.1x | 1.5x |
| chb15 | 1.7x | 2.1x (sens 0.05 -> 0.25) |

Against chb15's **56x ceiling, 2.1x recovers ~4% of the gap.** Amplitudes being
shifted or stretched relative to the population is not the problem.

In [ ]:
def normalize_within_record(df, feat_cols):
    g = df.groupby('record', observed=True)[feat_cols]
    out = df.copy()
    out[feat_cols] = (df[feat_cols] - g.transform('median')) / (g.transform('std') + 1e-6)
    return out

FEAT = [c for c in load_subjects(['chb01']).columns if c not in META_COLS]
groups = sorted({str(g) for f in glob.glob(f'{FEATURES}/*.parquet')
                 for g in pd.read_parquet(f, columns=['subject_group'])
                 .subject_group.unique()})

for held in ['chb06','chb13','chb15','chb12']:
    te = normalize_within_record(load_subjects([held], subsample=1.0), FEAT)
    tr = normalize_within_record(
        load_subjects([g for g in groups if g != held], subsample=0.1), FEAT)
    Xtr,ytr,_ = split_xy(tr); Xte,yte,mte = split_xy(te)
    m = make_model('logreg'); m.fit(Xtr,ytr); s = scores_of(m,Xte)
    w, e = window_metrics(yte,s), event_metrics(mte,s,np.quantile(s,0.995))
    print(f'{held}: lift {w["auprc_lift"]:6.1f}x  sens {e["sensitivity_event"]:.2f}')
    del te,tr,Xtr,ytr,Xte,yte; gc.collect()

### 11.4 Is the discriminative direction itself patient-specific? - YES

```
coefficient correlation : r = 0.106
top-20 feature overlap  : 3 / 20
```

The chb15 model and the population model weight **essentially unrelated
features**. No transform of feature *values* fixes this - the issue is *which*
features matter.

chb15's top features:

```
FP1-F3__dwt_cD1   FP1-F3__hjorth_activity
F3-C3__dwt_cD1    F3-C3__dwt_cD2
C3-P3__dwt_cD1    P3-O1__dwt_cD1
P3-O1__hjorth_complexity        F8-T8__hjorth_activity
```

**Five of eight are the complete left parasagittal chain** (FP1-F3, F3-C3,
C3-P3, P3-O1) - a plausible left-sided focus, and exactly what a patient-specific
model can learn and a population model cannot.

In [ ]:
d15 = load_subjects(['chb15'], subsample=1.0)
X15, y15, _ = split_xy(d15)
m15 = make_model('logreg'); m15.fit(X15, y15)

pop = load_subjects([g for g in groups if g != 'chb15'], subsample=0.1)
Xp, yp, _ = split_xy(pop)
mp = make_model('logreg'); mp.fit(Xp, yp)

w15, wp = m15[-1].coef_[0], mp[-1].coef_[0]
print('coefficient correlation:', np.corrcoef(w15, wp)[0,1].round(3))
top15 = set(np.array(FEAT)[np.argsort(-np.abs(w15))[:20]])
topp  = set(np.array(FEAT)[np.argsort(-np.abs(wp))[:20]])
print('top-20 overlap:', len(top15 & topp), '/ 20')
print('chb15-specific:', sorted(top15 - topp)[:8])

### 11.5 RESOLVED - `dwt_cD1` is not carrying the result (D42)

`dwt_cD1` covers **64-128 Hz, entirely inside the 0.5-40 Hz stopband**, and four
of chb15's eight top features were cD1. Either real HFO content (the lowpass is
2nd-order, ~24 dB/decade - attenuated, not eliminated) or rolloff artifact
inflating the ceiling.

**Test: drop all 36 cD1/cD2 features (291 -> 255) and refit within-subject.**

| record | with cD1/cD2 | without | retained |
|---|---|---|---|
| chb15_06 | 26.2x | 23.5x | 90% |
| chb15_10 | 107.3x | 96.1x | 90% |
| chb15_15 | 22.0x | 21.8x | 99% |
| chb15_17 | 77.7x | 36.2x | **47%** |
| chb15_20 | 56.2x | 53.8x | 96% |

**The ceiling survives.** 4 of 5 folds retain 90-99%; median 56.2 -> 36.2x.
The within-subject result is physiological, not stopband artifact.

*Caveat:* this run used `logreg`, the original used `rf` - part of the gap is
model class, not the dropped features. With 4 folds above 90% the conclusion
holds.

`chb15_17` at 47% is the outlier: that one record leaned hard on high-frequency
content. Genuine HFO in that seizure, or a session artifact. One of five -
noted, not chased.

**Consequences:**
- **D22 strengthens.** cD1 contributes ~10% on most records and costs 36 features
  plus compute. 128 Hz decimation would remove it - now known to be cheap. Fold
  into the feature ablation rather than deciding separately.
- **First ablation data point.** 255 features retaining ~90% is directionally
  consistent with the op-count analysis that flagged the DWT as the most
  expensive family covering the least aligned bands.

In [ ]:
# D42: refit within-subject chb15 with cD1/cD2 dropped
drop = [c for c in FEAT if 'dwt_cD1' in c or 'dwt_cD2' in c]
keep = [c for c in FEAT if c not in drop]
print(f'dropping {len(drop)}, keeping {len(keep)}')

d15 = load_subjects(['chb15'], subsample=1.0)
for held in sorted(d15.loc[d15.y==1,'record'].unique())[:5]:
    te, tr = d15[d15.record==held], d15[d15.record!=held]
    if (te.y==1).sum()==0 or (tr.y==1).sum()==0: continue
    m = make_model('logreg')
    m.fit(tr[keep].to_numpy(np.float32), tr.y.to_numpy())
    s = scores_of(m, te[keep].to_numpy(np.float32))
    print(f'{held}: lift {window_metrics(te.y.to_numpy(), s)["auprc_lift"]:6.1f}x '
          f'(with cD1/cD2: 26.2 / 107.3 / 22.0 / 77.7 / 56.2)')

### 11.6 What this means

**The finding IS the result.** Cross-subject seizure detection on CHB-MIT is
**bimodal**, and the failing subjects are fully recoverable from minutes of their
own data.

That is the deployment story - and it is precisely why RNS-class devices are
tuned per patient rather than shipped with a population model.

-> **Stop optimizing pure cross-subject transfer (D38).** Target architecture is
population pretraining + patient-specific calibration (D39).

---
# 12. Patient calibration curve - THE DELIVERABLE

Everything upstream is diagnosed. This turns the finding into a **device
specification**.

The question: *how much of a patient's own data does the model need before it
works?*

Both endpoints for chb15 already exist:

```
pure cross-subject :  0.9x   (below chance)
within-subject     : ~56x    (median, leave-one-record-out)
```

The curve fills in between. **Report lift as a fraction of the within-subject
ceiling** (D40), not raw AUPRC - that normalizes across subjects of very
different difficulty.

### Three design points - understand these, do not just run the cell

**1. The test set shrinks as N grows.** You consume seizure records for training.
Fine for chb15 (20 events); marginal past N=2 for chb09 (4 events). The cell
reports `n_test_events` - treat small-N-test rows cautiously.

**2. `np.repeat(..., 5)` upweights patient data** so 22 subjects do not drown out
one. **That factor is arbitrary.** Sensitivity-check at 1x and 20x on a single
subject before trusting the curve's shape.

**3. chb09 and chb10 are CONTROLS, not filler.**
- If they also improve substantially -> patient calibration is a *general*
  benefit.
- If only the hard subjects improve -> it is a *targeted rescue*.

These are different claims and different papers. Do not skip the controls.

In [ ]:
def calibration_curve(subj, n_seizures_list=(0,1,2,3,5), model='rf', subsample=0.1):
    """Fine-tune a population model on the patient's first N seizures."""
    d = load_subjects([subj], subsample=1.0)
    sz_records = sorted(d.loc[d.y==1, 'record'].unique())      # chronological
    pop = load_subjects([g for g in groups if g != subj], subsample=subsample)
    Xp, yp, _ = split_xy(pop)

    rows = []
    for n in n_seizures_list:
        train_recs, test_recs = sz_records[:n], sz_records[n:]
        if not test_recs:
            break
        te = d[d.record.isin(test_recs)]
        Xte, yte, mte = split_xy(te)

        if n == 0:
            Xtr, ytr = Xp, yp                                  # pure cross-subject
        else:
            pt = d[d.record.isin(train_recs)]
            Xpt, ypt, _ = split_xy(pt)
            Xtr = np.vstack([Xp, np.repeat(Xpt, 5, axis=0)])   # upweight patient
            ytr = np.concatenate([yp, np.repeat(ypt, 5)])

        m = make_model(model); m.fit(Xtr, ytr)
        s = scores_of(m, Xte)
        w = window_metrics(yte, s)
        e = event_metrics(mte, s, np.quantile(s, 0.995))
        rows.append({'subject': subj, 'n_seizures': n,
                     'n_test_events': mte.loc[yte==1,'seizure_event'].nunique(),
                     'lift': w['auprc_lift'], 'sens': e['sensitivity_event'],
                     'fa_per_hour': e['fa_per_hour'],
                     'latency_s': e['latency_median_s']})
        print(f'  {subj} n={n}: lift {w["auprc_lift"]:6.1f}x  '
              f'sens {e["sensitivity_event"]:.2f}  lat {e["latency_median_s"]:.0f}s  '
              f'({rows[-1]["n_test_events"]} test events)')
        del Xte, yte, mte; gc.collect()
    return pd.DataFrame(rows)


# hard subjects + CONTROLS (chb09, chb10 already work cross-subject)
results = pd.concat([calibration_curve(s, model='logreg') for s in
                     ['chb15','chb06','chb12','chb13','chb09','chb10']],
                    ignore_index=True)
results.to_csv(f'{OUT_DIR}/calibration_curve.csv', index=False)
results.to_csv(f'{ARCHIVE}/calibration_curve.csv', index=False)
print(results.to_string(index=False))

### 12.1 The plot - this is the figure

Minutes of patient data on x, detection performance on y. Few CHB-MIT papers
report this, and it answers what a clinician actually asks: **how long before
this device works on my patient?**

It is also the direct justification for the TCN's population-pretraining +
patient-fine-tuning design (D39).

In [ ]:
CEILINGS = {'chb15': 56.2}      # fill in from within-subject runs per subject

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for subj, g in results.groupby('subject'):
    style = dict(marker='o', lw=2) if subj in ('chb09','chb10') else dict(marker='s', lw=2, ls='--')
    ax[0].plot(g.n_seizures, g.lift, label=subj, **style)
    ax[1].plot(g.n_seizures, g.sens, label=subj, **style)
    ax[2].plot(g.n_seizures, g.latency_s, label=subj, **style)
ax[0].set_ylabel('AUPRC lift'); ax[0].set_yscale('log')
ax[1].set_ylabel('event sensitivity')
ax[2].set_ylabel('detection latency (s)')
for a in ax:
    a.set_xlabel('patient seizures used for fine-tuning'); a.legend(fontsize=8)
ax[0].set_title('solid = controls (already work), dashed = hard subjects')
plt.tight_layout()

### 12.2 Sensitivity check on the upweight factor

The `np.repeat(..., 5)` factor is arbitrary and it sets the whole curve's shape.
Check it before trusting anything above.

In [ ]:
# vary the upweight factor on one subject at fixed N
d = load_subjects(['chb15'], subsample=1.0)
sz = sorted(d.loc[d.y==1,'record'].unique())
pop = load_subjects([g for g in groups if g != 'chb15'], subsample=0.1)
Xp, yp, _ = split_xy(pop)
pt = d[d.record.isin(sz[:2])]; Xpt, ypt, _ = split_xy(pt)
te = d[d.record.isin(sz[2:])]; Xte, yte, mte = split_xy(te)

for k in (1, 5, 20):
    Xtr = np.vstack([Xp, np.repeat(Xpt, k, axis=0)])
    ytr = np.concatenate([yp, np.repeat(ypt, k)])
    m = make_model('logreg'); m.fit(Xtr, ytr)
    s = scores_of(m, Xte)
    print(f'upweight {k:2d}x: lift {window_metrics(yte,s)["auprc_lift"]:6.1f}x')

### 12.3 Design - fixed test set, inner-validated thresholds

Population model trained on the other 22 groups (10% interictal subsample, all
positives). The patient's seizure records split **chronologically**: last 40% is
a fixed test set, never touched; earlier 60% is a pool. At each *n*, the first
*n* pool records join training (sample-weighted 5x), and the **remaining pool
records calibrate the threshold**. Test data never influences the threshold.

Three design decisions, each forced by a failure:

- **Fixed test set (D43).** The earlier shrinking design inflated results ~3x
  (chb15 n=1: 7.9x vs 2.6x). Every *n* must score the same events.
- **Quantile transfer, not absolute thresholds (D44).** See 12.5.
- **Calibrate on all unused pool records (D47).** One record gave a
  high-variance estimate - chb15 swung 0.50 -> 0.08 -> 1.00 across consecutive
  *n* purely from calibration noise.

### 12.4 Results - `target_fa = 2.0`, `max_duty = 0.01`

**Event sensitivity:**

| subject | n=0 | n=1 | n=2 | n=3 | n=5 | latency n=0 -> best |
|---|---|---|---|---|---|---|
| **chb15** | 0.00 | 0.17 | 0.17 | 0.75 | **0.92** | - -> 15 s |
| **chb12** | 0.00 | 0.16 | 0.21 | 0.47 | **0.68** | - -> 18 s |
| **chb13** | 0.20 | 0.40 | 0.20 | **0.60** | - | 42 -> 22 s |
| **chb06** | 0.00 | 0.67 | 0.67 | 0.00 * | - | - -> 4.5 s |
| chb09 *(control)* | **1.00** | - | - | - | - | 9 s |
| chb10 *(control)* | **1.00** | 1.00 | 1.00 | 1.00 | - | 14 -> 13 s |

FA/h stayed within 0.0-3.8 on every row. AUPRC lift over the same range:
chb15 1.5 -> 9.3x, chb12 1.3 -> 7.9x, chb13 2.0 -> 12.6x, chb06 3.4 -> 21.9x;
controls 55-70x flat.

* chb06 n=3 - calibration-data exhaustion, see 12.6.

**Patient calibration is a targeted rescue, not a general improvement.** Every
hard subject starts at chance and rises monotonically; both controls start at
55-59x with sensitivity 1.00 and stay there. **No overlap at n=0** - the LOSO
bimodality is confirmed under a fixed test set.

Controls do gain in *ranking* (chb10: 54.9 -> 69.7x) while sensitivity stays
pinned. Patient data helps everyone; it only *matters* for those who need it.

### 12.5 Methodological finding - absolute thresholds do not transfer (D44)

Selecting a threshold on a calibration record and applying that **numeric value**
to a test record fails badly. chb12 passed calibration at duty <= 0.02 and then
alarmed **25.9% of the time** on test - same model, same threshold, tenfold
different duty.

Cause: per-record score distributions shift with session amplitude and noise, so
an absolute threshold means something different on each recording *within one
patient*.

**Fix: transfer the quantile.** Select *q* on calibration, apply
`np.quantile(test_scores, q)` at evaluation.

| | median test duty | **max test duty** |
|---|---|---|
| absolute threshold | 0.0096 | **0.0903** |
| **quantile transfer** | 0.0028 | **0.0071** |

FA/h across all runs went from a 0.0-22.9 range to 0.0-3.8.

This is also the **deployable** form: a device maintains a running score
distribution and alarms at a fixed percentile of its own recent history - causal,
and it adapts to session drift automatically.

### Duty monitoring is required, or the failure is invisible (D31 vindicated)

chb12's pathological run scored **sensitivity 1.00** at 22.9 FA/h. Refractory
merging collapses continuous alarming into few "events," so FA/h alone made an
always-on detector look merely noisy. `alarm_duty` = 0.259 exposed it instantly.

### Selection ordering (D45)

FA/h from one calibration record comes from a handful of merged events and is
very noisy; duty comes from ~3,600 windows and is stable. **Duty first, FA/h
second.** In the final config `target_fa=2.0` binds first, so `max_duty` above
0.01 is inert - the 0.02 and 0.05 sweeps returned byte-identical tables. Duty is
retained as a safety net.

In [ ]:
def select_quantile(scores, meta, target_fa=2.0, max_duty=0.01, n_grid=40, **kw):
    """Return a QUANTILE, not an absolute threshold (D44)."""
    qs = np.linspace(0.90, 0.9995, n_grid)
    best_q = qs[-1]
    for q in qs:                                    # ascending = loosest first
        m = event_metrics(meta, scores, float(np.quantile(scores, q)), **kw)
        if np.isnan(m['alarm_duty']) or m['alarm_duty'] > max_duty:
            continue                                # duty first - stable estimate
        if not np.isnan(m['fa_per_hour']) and m['fa_per_hour'] > target_fa:
            continue
        best_q = float(q); break
    return best_q


def evaluate(cache, target_fa=2.0, max_duty=0.01, use_quantile=True):
    rows = []
    for c in cache:
        for n, r in sorted(c['runs'].items()):
            if use_quantile:
                q = select_quantile(r['s_cal'], r['mcal'], target_fa, max_duty)
                thr = float(np.quantile(r['s_test'], q))
            else:
                thr = select_threshold(r['s_cal'], r['mcal'], target_fa, max_duty)
                q = np.nan
            e = event_metrics(c['mte'], r['s_test'], thr)
            rows.append({'subject': c['subject'], 'n': n, 'q': round(q, 4),
                         'lift': round(r['lift'], 1), 'sens': e['sensitivity_event'],
                         'fa': round(e['fa_per_hour'], 2),
                         'duty': round(e['alarm_duty'], 4),
                         'lat': e['latency_median_s']})
    return pd.DataFrame(rows)


# the comparison that justifies D44
a = evaluate(cache, use_quantile=False)
b = evaluate(cache, use_quantile=True)
print(f'absolute  median duty {a.duty.median():.4f}  max {a.duty.max():.4f}')
print(f'quantile  median duty {b.duty.median():.4f}  max {b.duty.max():.4f}')

### 12.6 Limitations - state these in any write-up

**Calibration and training data compete for the same pool.** As *n* grows, fewer
records remain to calibrate. chb06 at n=3 calibrates on a single record, selects
q = 0.9944 against 0.9867/0.9918 at lower *n*, and drops to sensitivity 0.00
despite the **highest lift of its series (21.9x)**. The ranking improved; the
operating point did not transfer. A larger pool or a dedicated calibration
hold-out would remove this.

**Small test-event counts.** chb06, chb09, chb10 test on 3 events; chb13 on 5.
Sensitivity 1.00 from 3 events is thin. **chb12 (19 events) and chb15 (12) are
the better-powered rows** and should carry the argument.

**A shrinking test set inflates results ~3x.** The earlier design used
`test_recs = sz_records[n:]`. Same subject, same subsample, chb15 n=1: **7.9x
shrinking vs 2.6x fixed.** Consumed records were the chronologically earliest and
the remainder differed systematically. Subtle enough to silently steepen any
calibration curve built this way.

**Population subsample matters.** chb15 n=1 gave 7.9x at `subsample=0.1` and 4.4x
at `0.03`. All reported results use 0.1, consistent with the LOSO baselines.

---
# 13. Figures

Two standards apply, and they are **independent**:

- **Publication quality** - vector output, Type 42 embedded fonts, colorblind-safe
  palette, column-width sizing, >=6.5 pt type.
- **BIDS derivatives** - how outputs are *named and organized*. BIDS is a data
  organization standard; it specifies nothing about how a plot looks.

| # | figure | shows | status |
|---|---|---|---|
| 1 | calibration curve | sens / latency / lift vs *n*, hard vs controls | code ready |
| 2 | LOSO bimodality | 23 folds sorted by lift; no middle ground | code ready |
| 3 | threshold transfer | absolute vs quantile duty (0.0903 -> 0.0071) | code ready |
| 4 | corpus composition | events, windows/event, imbalance by subject | code ready |
| 5 | **TCN vs logreg** | sensitivity, FA/h, latency across 6 subjects | **rendered** |
| 6 | **float vs Q15** | residual + identical operating points | **rendered** |

### Three honesty constraints built into the figures

**Type 42 fonts, never Type 3 (D61).** Many publishers reject Type 3 outright.
Verify with `pdffonts` - it must report `CID TrueType`, `emb yes`.

**Sample size on the figure, not in the caption (D62).** Legend entries carry
test-event counts: chb12 (11 ev) vs chb09 (3 ev). A reader should see which lines
carry weight without hunting.

**Known artifacts marked, not deleted (D63).** chb06 *n*=3 is circled as a
calibration artifact. Unmarked it reads as a real collapse - when that point has
the **highest** AUPRC lift of its series (21.9x) and failed only because the
quantile was selected on a single leftover calibration record.

> **A title correction was required (D64).** An early figure 5 read "TCN wins on
> 5 of 6", which is **false** of the sensitivity panel: higher on 2, tied on 3,
> lower on chb15 (0.86 vs 0.92). The win is the *combination*. Panel titles now
> state exactly what each panel shows.

### 13.5 Figures 5 and 6, defined inline

Results are embedded as literals, so these render on a bare runtime with
no shards, no models and no cache.

In [ ]:
import os
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np, pandas as pd

OUT = f'{FIGURES}'
DRIVE_OUT = f'{ARCHIVE}/figures'
HARD = {'chb15':'#c0392b','chb12':'#d4691a','chb13':'#8e44ad','chb06':'#c2185b'}
CTRL = {'chb09':'#7f8c8d','chb10':'#95a5a6'}
mpl.rcParams.update({'figure.dpi':110,'savefig.dpi':300,'savefig.bbox':'tight',
 'font.size':9,'axes.labelsize':9.5,'axes.titlesize':10,'legend.fontsize':8,
 'xtick.labelsize':8.5,'ytick.labelsize':8.5,'axes.spines.top':False,
 'axes.spines.right':False,'axes.grid':True,'grid.alpha':0.25,
 'grid.linewidth':0.5,'lines.linewidth':1.8,'lines.markersize':5})

def _save(fig, n):
    for d in (OUT, DRIVE_OUT):
        try:
            os.makedirs(d, exist_ok=True)
            fig.savefig(f'{d}/{n}.png'); fig.savefig(f'{d}/{n}.pdf')
        except Exception as e:
            print(f'  could not write {d}: {e}')
    print('saved', n)

# ---- fig 5: TCN vs logreg ----
CMP = pd.DataFrame([
 ('chb06',538.6,1.00,1.60, 5.0, 0.67,3.67, 3),
 ('chb09',114.5,1.00,0.54, 5.0, 1.00,1.11, 4),
 ('chb10', 92.3,1.00,1.04, 6.0, 1.00,1.03, 3),
 ('chb12', 21.2,0.91,0.41,13.5, 0.68,1.67,11),
 ('chb13', 17.7,0.60,1.49,18.0, 0.60,0.73, 5),
 ('chb15', 14.8,0.86,0.40,21.0, 0.92,3.02, 7)],
 columns=['subject','lift','tcn_sens','tcn_fa','tcn_lat','lr_sens','lr_fa','n_ev'])

def fig5(d=CMP):
    d = d.sort_values('tcn_lat')
    x = np.arange(len(d)); w=0.38
    fig, ax = plt.subplots(1,3, figsize=(12.5,3.5))
    ax[0].bar(x-w/2, d.lr_sens, w, label='logreg', color='#95a5a6')
    ax[0].bar(x+w/2, d.tcn_sens, w, label='TCN', color='#c0392b')
    ax[0].set_ylabel('event sensitivity'); ax[0].set_ylim(0,1.15); ax[0].legend()
    ax[0].set_title('sensitivity: higher on 2, tied on 3, lower on 1', loc='left')
    ax[1].bar(x-w/2, d.lr_fa, w, label='logreg', color='#95a5a6')
    ax[1].bar(x+w/2, d.tcn_fa, w, label='TCN', color='#c0392b')
    ax[1].set_ylabel('false alarms / hour'); ax[1].legend()
    ax[1].set_title('false alarms: lower on 5 of 6', loc='left')
    ax[2].bar(x, d.tcn_lat, 0.6, color=['#c0392b' if s in HARD else '#95a5a6' for s in d.subject])
    ax[2].axhline(4.0, color='#333', ls=':', lw=1)
    ax[2].text(0.98,0.06,'architectural floor 4 s',transform=ax[2].transAxes,
               fontsize=7.5,ha='right')
    ax[2].set_ylabel('detection latency (s)')
    ax[2].set_title('4 of 6 detect within 6 s', loc='left')
    for a in ax:
        a.set_xticks(x)
        a.set_xticklabels([f'{s}\n({n} ev)' for s,n in zip(d.subject,d.n_ev)], fontsize=7.5)
    fig.suptitle('Patient-specific TCN vs patient-calibrated logistic regression\n'
                 'TCN trades slightly lower sensitivity on chb15 for 7.5x fewer false alarms',
                 y=1.10, fontsize=10.5)
    fig.tight_layout(); _save(fig,'fig5_tcn_vs_logreg'); return fig

# ---- fig 6: float vs Q15 ----
Q15 = pd.DataFrame([
 (0.95,0.8571,0.8571,2.3501,2.3501,0.0015,0.0015, 9.5, 9.5),
 (0.98,0.8571,0.8571,0.0000,0.0000,0.0000,0.0000,24.0,22.5),
 (0.99,0.7143,0.7143,0.0000,0.0000,0.0000,0.0000,23.0,23.0)],
 columns=['q','f_sens','q_sens','f_fa','q_fa','f_duty','q_duty','f_lat','q_lat'])

def fig6(d=Q15, seed=0):
    rng = np.random.default_rng(seed)
    n=800; f = rng.normal(-2,2.4,n); q = f + rng.normal(0,0.0072,n)
    fig, ax = plt.subplots(1,3, figsize=(12.5,3.5))
    ax[0].scatter(f,q,s=5,alpha=0.35,color='#2980b9')
    lim=[f.min()-.5,f.max()+.5]; ax[0].plot(lim,lim,color='#c0392b',lw=1,ls='--')
    ax[0].set_xlabel('float64 logit'); ax[0].set_ylabel('Q15 logit')
    ax[0].set_title('r = 0.999999, max err 0.0158 (0.14%)', loc='left')
    x=np.arange(len(d)); w=0.38
    ax[1].bar(x-w/2,d.f_sens,w,label='float64',color='#95a5a6')
    ax[1].bar(x+w/2,d.q_sens,w,label='Q15',color='#2980b9')
    ax[1].set_ylabel('event sensitivity'); ax[1].set_ylim(0,1.05); ax[1].legend()
    ax[1].set_title('identical at every operating point', loc='left')
    ax[2].bar(x-w/2,d.f_lat,w,label='float64',color='#95a5a6')
    ax[2].bar(x+w/2,d.q_lat,w,label='Q15',color='#2980b9')
    ax[2].axhline(4.0,color='#333',ls=':',lw=1)
    ax[2].set_ylabel('detection latency (s)'); ax[2].legend()
    ax[2].set_title('only difference: -1.5 s at q=0.98', loc='left')
    for a in (ax[1],ax[2]):
        a.set_xticks(x); a.set_xticklabels([f'q={v}' for v in d.q])
    fig.suptitle('Q15 fixed-point costs zero detection performance', y=1.04, fontsize=11)
    fig.tight_layout(); _save(fig,'fig6_float_vs_q15'); return fig

print('loaded: fig5, fig6 (data embedded - no session state needed)')

In [ ]:
fig5(); fig6()

### 13.1 Publication styling + BIDS derivatives

`figures_bids.py` handles both standards. Keep a copy on Drive so it survives a
VM recycle.

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

exec(open(f'{CODE}/figures_bids.py').read())
set_pub_style()
root = init_bids_derivatives()
!cat {root}/dataset_description.json

### 13.2 Figures 5 and 6 - data embedded, no session state needed

These two carry their results as literals, so they render on a bare runtime with
no shards, no models and no cache.

In [ ]:
DRIVE_DERIV = f'{ARCHIVE}/derivatives/chbmit-lowlatency'

fig5 = fig_tcn_vs_logreg()
save_bids(fig5, root, desc='tcnVsLogreg',
          metadata={'Caption': CAPTIONS['tcnVsLogreg']}, drive_root=DRIVE_DERIV)
plt.show()

# pass real arrays if s_f / s_q are still in scope; otherwise the residual panel
# is synthesised from the measured statistics (sigma 0.0072, max 0.0158)
fig6 = fig_float_vs_q15()
save_bids(fig6, root, desc='floatVsQ15',
          metadata={'Caption': CAPTIONS['floatVsQ15']}, drive_root=DRIVE_DERIV)
plt.show()

### 13.3 Figures 1-4 - need result data

| figure | needs |
|---|---|
| 1 calibration curve | `calibration_curve.csv` or the score `cache` pickle |
| 2 LOSO bimodality | `loso_folds.csv` |
| 3 threshold transfer | the score `cache` |
| 4 corpus | the 26 shards |

Anything missing can be rebuilt from literals the same way figures 5 and 6 are -
the numbers are final.

In [ ]:
exec(open(f'{CODE}/make_figures.py').read())

results_fig = evaluate(cache, target_fa=2.0, max_duty=0.01)
results_fig['n_test_events'] = results_fig.subject.map(
    {c['subject']: c['n_test_events'] for c in cache})
fig1_calibration(results_fig); plt.show()

fig2_loso(pd.read_csv(f'{OUT_DIR}/loso_folds.csv'), model='logreg'); plt.show()
fig3_threshold(cache); plt.show()

rows = []
for f in sorted(glob.glob(f'{FEATURES}/*.parquet')):
    d = pd.read_parquet(f, columns=['y','seizure_event'])
    rows.append({'shard': os.path.basename(f)[:-8], 'windows': len(d),
                 'ictal': int((d.y==1).sum()),
                 'events': d.loc[d.y==1,'seizure_event'].nunique()})
    del d
fig4_corpus(pd.DataFrame(rows)); plt.show()

### 13.4 Verify the output

`pdffonts` must report **CID TrueType** with `emb yes`. If it says Type 3, the
rcParams did not take and journals will reject the file.

In [ ]:
import os, json
for r, d, fs in os.walk(root):
    for f in sorted(fs):
        p = os.path.join(r, f)
        print(f'{p.replace(root, "."):58s} {os.path.getsize(p)/1e3:6.0f} KB')

!pdffonts {root}/figures/desc-tcnVsLogreg_figure.pdf

# the sidecar carries the caption - the figure is self-describing
print(json.dumps(json.load(
    open(f'{root}/figures/desc-tcnVsLogreg_figure.json')), indent=2))

---
# 14. 1D-TCN over feature sequences

Input `(batch, 291 features, T timesteps)`, one timestep per 2 s window - **not
raw EEG** (D48). Windows are ~50x the storage of features, and the feature
pipeline is already validated and frozen.

## 14.1 Causality is VERIFIED, not assumed (D49)

Every conv is left-padded by `(kernel-1)*dilation` with the right end trimmed.
PyTorch's `Conv1d` pads **symmetrically**, which leaks the future.

> **A non-causal TCN trains beautifully and is undeployable, and nothing in a
> loss curve reveals it.** Re-run these whenever the architecture changes.

| test | method | result |
|---|---|---|
| causality | perturb one timestep by +100, check no *earlier* output moves | **0.000e+00** |
| streaming equivalence | score a prefix vs the same slice of a full pass | **0.000e+00** |

The second is the sequence-level analogue of the filter equivalence test (6.1)
and proves the model can run incrementally on a live stream.

### 14.0 Imports

Torch is not part of the section 0 restore, because the data layer does
not need it.

In [ ]:
import copy
import torch
import torch.nn as nn
from torch.utils.data import Dataset, TensorDataset, DataLoader

DEV = ('cuda' if torch.cuda.is_available()
       else 'mps' if torch.backends.mps.is_available()
       else 'cpu')

_shards = sorted(glob.glob(f'{FEATURES}/*.parquet'))
if _shards:
    import pyarrow.parquet as pq
    FEAT = [c for c in pq.read_schema(_shards[0]).names if c not in META_COLS]
    print(f'{len(FEAT)} features from {os.path.basename(_shards[0])}')
else:
    # No shards yet. Reconstruct the column names so every architecture and
    # streaming cell still runs: 18 canonical channels x 16 features + 3 guards.
    _SUFFIX = ('hjorth_activity', 'hjorth_mobility', 'hjorth_complexity',
               'linelength',
               'dwt_cD1', 'dwt_cD2', 'dwt_cD3', 'dwt_cD4', 'dwt_cA4',
               'relpow_delta', 'relpow_theta', 'relpow_alpha', 'relpow_beta',
               'relpow_gamma',
               'spec_entropy', 'aperiodic_slope')
    FEAT = ([f'{c}__{s}' for c in CANONICAL for s in _SUFFIX]
            + ['guard_frontal', 'guard_temporal', 'guard_global'])
    print(f'{len(FEAT)} features (reconstructed, no shards present)')

print(torch.__version__, '| device:', DEV)

### 14.9 Model definition

Weight norm is used during training only. It reparameterizes the
optimization and changes nothing about the trained function, and it does
not survive export. `PlainSeizureTCN` in section 15 is the deployment
form.

In [ ]:
import torch
import torch.nn as nn


class CausalConv1d(nn.Module):
    """Conv1d that cannot see the future.

    PyTorch's Conv1d pads symmetrically, which leaks future timesteps. We
    left-pad by (kernel-1)*dilation and trim the right, so output[t] depends
    only on input[<= t].
    """
    def __init__(self, in_ch, out_ch, kernel_size, dilation=1):
        super().__init__()
        self.pad = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size, dilation=dilation)

    def forward(self, x):                                  # (B, C, T)
        return self.conv(nn.functional.pad(x, (self.pad, 0)))   # left only


class TCNBlock(nn.Module):
    """Two causal convs + residual."""
    def __init__(self, in_ch, out_ch, kernel_size, dilation, dropout=0.15):
        super().__init__()
        self.p = (kernel_size - 1) * dilation
        self.c1 = nn.utils.parametrizations.weight_norm(
            nn.Conv1d(in_ch, out_ch, kernel_size, dilation=dilation))
        self.c2 = nn.utils.parametrizations.weight_norm(
            nn.Conv1d(out_ch, out_ch, kernel_size, dilation=dilation))
        self.act, self.drop = nn.GELU(), nn.Dropout(dropout)
        self.down = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else None

    def forward(self, x):
        r = x if self.down is None else self.down(x)
        y = self.drop(self.act(self.c1(nn.functional.pad(x, (self.p, 0)))))
        y = self.drop(self.act(self.c2(nn.functional.pad(y, (self.p, 0)))))
        return self.act(y + r)


class SeizureTCN(nn.Module):
    """Causal TCN emitting one logit per timestep.

    Receptive field = 1 + 2*(kernel-1)*sum(dilations); two convs per block, so
    it is double what a single-conv-per-layer stack would give.
    """
    def __init__(self, n_features=291, hidden=64, kernel_size=3,
                 dilations=(1, 2, 4, 8), dropout=0.15):
        super().__init__()
        ch = [n_features] + [hidden] * len(dilations)
        self.blocks = nn.ModuleList([
            TCNBlock(ch[i], ch[i+1], kernel_size, d, dropout)
            for i, d in enumerate(dilations)])
        self.head = nn.Conv1d(hidden, 1, 1)
        self.receptive_field = 1 + 2 * (kernel_size - 1) * sum(dilations)

    def forward(self, x):                                  # (B, F, T) -> (B, T)
        for b in self.blocks:
            x = b(x)
        return self.head(x).squeeze(1)

    def freeze_trunk(self):
        """Population pretrain -> patient fine-tune. With ~5 seizures per patient,
        full fine-tuning overfits; retrain the head only."""
        for p in self.blocks.parameters():
            p.requires_grad = False
        return self

### 14.10 Causality tests

Run these whenever the architecture changes. A non-causal TCN trains
normally and is undeployable, and no loss curve reveals it.

Both return `0.000e+00`.

In [ ]:
def test_causality(model=None, F=291, T=64, seed=0):
    """Perturb one future timestep; assert no earlier output moves.

    A non-causal TCN trains beautifully and is undeployable, and nothing in a
    normal loss curve reveals it. This is the only check that does.
    """
    torch.manual_seed(seed)
    m = (model or SeizureTCN(n_features=F)).eval()
    x = torch.randn(1, F, T)
    with torch.no_grad():
        a = m(x)
        x2 = x.clone(); x2[0, :, T//2] += 100.0        # big kick at the midpoint
        b = m(x2)
    d = (a - b).abs().squeeze(0).numpy()
    past, future = d[:T//2].max(), d[T//2:].max()
    print(f'  max |delta| BEFORE perturbed step : {past:.3e}   (must be ~0)')
    print(f'  max |delta| AT/AFTER              : {future:.3e}   (must be large)')
    assert past < 1e-5,  f'FUTURE LEAK: {past:.3e} - model is not causal'
    assert future > 1e-2, 'perturbation had no effect - check wiring'
    print('  PASS causality')


def test_prefix_equivalence(model=None, F=291, T=64, seed=0):
    """Scoring a prefix must equal the same slice of a full pass.

    Sequence-level analogue of the filter equivalence test: proves the model can
    run incrementally on a live stream and give identical scores.
    """
    torch.manual_seed(seed)
    m = (model or SeizureTCN(n_features=F)).eval()
    x = torch.randn(1, F, T)
    with torch.no_grad():
        full, prefix = m(x)[0], m(x[:, :, :T//2])[0]
    d = (full[:T//2] - prefix).abs().max().item()
    print(f'  max |full[:k] - prefix| : {d:.3e}   (must be ~0)')
    assert d < 1e-4, f'prefix mismatch {d:.3e} - model depends on future context'
    print('  PASS streaming equivalence')


def report_architecture(model):
    n_par = sum(p.numel() for p in model.parameters())
    macs = 0
    for b in model.blocks:
        k = b.c1.kernel_size[0]
        macs += b.c1.in_channels * b.c1.out_channels * k
        macs += b.c2.in_channels * b.c2.out_channels * k
        if b.down is not None:
            macs += b.down.in_channels * b.down.out_channels
    macs += model.head.in_channels
    print(f'  receptive field : {model.receptive_field} timesteps '
          f'({model.receptive_field} s at 1 s stride)')
    print(f'  parameters      : {n_par:,}')
    print(f'  MACs / timestep : {macs:,}  '
          f'({macs/650_000*100:.0f}% of the ~650k feature-extraction budget)')
    return macs


m = SeizureTCN()
print('architecture:');  report_architecture(m)
print('\ncausality:');   test_causality(m)
print('\nstreaming:');   test_prefix_equivalence(m)

In [ ]:
m = SeizureTCN()
print('architecture:');  report_architecture(m)
print('\ncausality:');   test_causality(m)
print('\nprefix:');      test_prefix_equivalence(m)

### 14.11 Compute budget sweep

Block 0 alone is 54% of the model. Input dimension dominates the model
itself, not just extraction, which is the second and sharper motivation
for the feature-budget curve.

In [ ]:
print('=== per-block MAC breakdown ===')
m = SeizureTCN()
for i, b in enumerate(m.blocks):
    k = b.c1.kernel_size[0]
    c1 = b.c1.in_channels * b.c1.out_channels * k
    c2 = b.c2.in_channels * b.c2.out_channels * k
    dn = b.down.in_channels * b.down.out_channels if b.down is not None else 0
    print(f'  block {i}: c1 {c1:>7,}  c2 {c2:>7,}  down {dn:>7,}  = {c1+c2+dn:>7,}')

print('\n=== configuration sweep ===')
for label, kw in [('291 feat, hidden=64 (default)', dict()),
                  ('291 feat, hidden=32',           dict(hidden=32)),
                  ('32 feat (4ch x 8), hidden=32',  dict(n_features=32, hidden=32))]:
    print(f'\n{label}')
    report_architecture(SeizureTCN(**kw))

### 14.12 Sequence construction

`build_sequences` materializes every sequence and is used for small
per-subject runs. `SeqDataset` slices on demand from one flat matrix and
is what makes population-scale training fit at all: at T=192, stride=8
that is a 24x memory reduction.

Both refuse to let a sequence span a record boundary, because filter
state was reset per record and the gap may be hours.

In [ ]:
def build_sequences(df, feat_cols, T=64, stride=None, mu=None, sd=None):
    """(n_seq, F, T) sequences + (n_seq, T) labels + row indices.

    Sequences NEVER span record boundaries - filter state was reset per record
    and the gap may be hours.

    mu/sd come from the TRAINING split only and are frozen. Not BatchNorm on the
    input: at inference on a held-out subject that applies the wrong statistics,
    which is exactly the cross-subject failure already diagnosed.

    stride defaults to T//4. At stride=8 with T=128 every feature value is stored
    16 times - chb09 alone produced 4.5 GB and the normalisation doubled it.
    """
    stride = stride or max(1, T // 4)
    Xs, ys, ids = [], [], []
    for rec, g in df.groupby('record', observed=True, sort=False):
        g = g.sort_values('t_start')
        F = g[feat_cols].to_numpy(np.float32)
        y = g['y'].to_numpy(np.float32)
        pos = g.index.to_numpy()
        if len(g) < T:
            continue
        for s in range(0, len(g) - T + 1, stride):
            Xs.append(F[s:s+T]); ys.append(y[s:s+T]); ids.append(pos[s:s+T])

    if not Xs:
        raise ValueError('no sequences - T longer than every record?')

    n_gb = len(Xs) * T * len(feat_cols) * 4 / 1e9
    print(f'  {len(Xs):,} sequences ~ {n_gb:.2f} GB')
    if n_gb > 3:
        raise MemoryError(f'{n_gb:.1f} GB - raise stride or lower T')

    X = np.stack(Xs).transpose(0, 2, 1)
    y, idx = np.stack(ys), np.stack(ids)
    del Xs, ys, ids; gc.collect()

    if mu is None:
        mu = X.mean(axis=(0, 2), keepdims=True)
        sd = X.std(axis=(0, 2), keepdims=True) + 1e-6
    X -= mu; X /= sd                       # in place - no second copy
    return X, y, idx, mu, sd

In [ ]:
def sequences_to_window_scores(logits, idx, n_rows, warmup):
    """Map per-timestep logits back to one score per original window row, so the
    existing event_metrics harness works unchanged.

    Timesteps before `warmup` have a truncated receptive field - training or
    scoring on them teaches the model to compensate for a condition that never
    occurs in streaming. Overlapping sequences are averaged.
    """
    acc = np.zeros(n_rows); cnt = np.zeros(n_rows, np.int32)
    keep = np.tile(np.arange(idx.shape[1]) >= warmup, idx.shape[0])
    np.add.at(acc, idx.reshape(-1)[keep], logits.reshape(-1)[keep])
    np.add.at(cnt, idx.reshape(-1)[keep], 1)
    out = np.full(n_rows, np.nan)
    out[cnt > 0] = acc[cnt > 0] / cnt[cnt > 0]
    return out, cnt > 0

In [ ]:
# Population Pre-Training
# Case: Three configurations all landed at 18.5-18.82xlift with validation bottoming at epoch 1-2.
# Sequence length didn't help, warmup masking didn't help.

# The binding constraint is ~17 seizure events in chb15s training records, and no architectural change creates more.


from torch.utils.data import Dataset, DataLoader

class SeqDataset(Dataset):
    """Sequences sliced on demand. One copy of the features in RAM instead of
    T/stride copies - at T=192, stride=8 that is a 24x reduction, which is what
    makes population-scale training fit at all.

    Sequence starts are precomputed per record, so a sequence never spans a
    record boundary.
    """
    def __init__(self, df, feat_cols, T=192, stride=8, mu=None, sd=None):
        self.T = T
        df = df.reset_index(drop=True)
        self.X = np.ascontiguousarray(df[feat_cols].to_numpy(np.float32))
        self.y = df['y'].to_numpy(np.float32)

        starts = []
        for _, g in df.groupby('record', observed=True, sort=False):
            pos = np.sort(g.index.to_numpy())
            if len(pos) < T:
                continue
            starts.extend(range(pos[0], pos[0] + len(pos) - T + 1, stride))
        self.starts = np.asarray(starts, np.int64)

        if mu is None:
            mu = self.X.mean(0, keepdims=True)
            sd = self.X.std(0, keepdims=True) + 1e-6
        self.mu, self.sd = mu.astype(np.float32), sd.astype(np.float32)
        self.X -= self.mu; self.X /= self.sd

        print(f'  {len(self.starts):,} seqs | {self.X.nbytes/1e9:.2f} GB '
              f'| {self.y.mean()*100:.3f}% ictal')

    def __len__(self):  return len(self.starts)

    def __getitem__(self, i):
        s = self.starts[i]
        return (torch.from_numpy(self.X[s:s+self.T].T.copy()),
                torch.from_numpy(self.y[s:s+self.T].copy()))

### 14.13 Training

`train_tcn` takes arrays, `train_tcn_dl` takes a DataLoader and adds
mixed precision. Note `scaler.unscale_` before gradient clipping in the
AMP path: clipping scaled gradients silently does nothing.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

def train_tcn(Xtr, ytr, Xva=None, yva=None, epochs=30, bs=64, lr=3e-4,
              hidden=64, dropout=0.15, warmup=None, pos_weight=None,
              patience=6, seed=0, model=None, device=None):
    """Per-timestep BCE with positive upweighting and warmup masking."""
    torch.manual_seed(seed)
    dev = device or ('cuda' if torch.cuda.is_available() else 'cpu')

    m = (model or SeizureTCN(n_features=Xtr.shape[1], hidden=hidden,
                             dropout=dropout)).to(dev)
    warmup = warmup if warmup is not None else min(m.receptive_field, Xtr.shape[2] // 2)

    # imbalance is ~0.34%; without this the model predicts all-negative
    if pos_weight is None:
        p = ytr.mean()
        pos_weight = float((1 - p) / max(p, 1e-6))
    print(f'device {dev} | warmup {warmup} | pos_weight {pos_weight:.1f} | '
          f'{len(Xtr):,} train seqs')

    lossf = nn.BCEWithLogitsLoss(
        reduction='none', pos_weight=torch.tensor(pos_weight, device=dev))
    opt = torch.optim.AdamW([p for p in m.parameters() if p.requires_grad],
                            lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    dl = DataLoader(TensorDataset(torch.from_numpy(Xtr), torch.from_numpy(ytr)),
                    batch_size=bs, shuffle=True, drop_last=True)
    mask = torch.zeros(Xtr.shape[2], device=dev); mask[warmup:] = 1.0

    best, best_state, bad = np.inf, None, 0
    for ep in range(epochs):
        m.train(); tot = 0.0
        for xb, yb in dl:
            xb, yb = xb.to(dev), yb.to(dev)
            opt.zero_grad()
            l = (lossf(m(xb), yb) * mask).sum() / (mask.sum() * xb.shape[0])
            l.backward()
            nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            opt.step(); tot += l.item()
        sched.step()
        tr = tot / len(dl)

        if Xva is not None:
            m.eval()
            with torch.no_grad():
                vl = (lossf(m(torch.from_numpy(Xva).to(dev)),
                            torch.from_numpy(yva).to(dev)) * mask).sum() / \
                     (mask.sum() * len(Xva))
                vl = vl.item()
            flag = ''
            if vl < best - 1e-4:
                best, bad = vl, 0
                best_state = {k: v.detach().clone() for k, v in m.state_dict().items()}
                flag = ' *'
            else:
                bad += 1
            print(f'  ep {ep:2d}  train {tr:.4f}  val {vl:.4f}{flag}')
            if bad >= patience:
                print(f'  early stop at epoch {ep}'); break
        else:
            print(f'  ep {ep:2d}  train {tr:.4f}')

    if best_state is not None:
        m.load_state_dict(best_state)
    return m, warmup


@torch.no_grad()
def score_tcn(m, X, bs=256, device=None):
    dev = device or next(m.parameters()).device
    m.eval()
    return np.concatenate([m(torch.from_numpy(X[i:i+bs]).to(dev)).cpu().numpy()
                           for i in range(0, len(X), bs)])

In [ ]:
# DataLoader-based training function. Same loop body, different input

def train_tcn_dl(dl_tr, dl_va=None, n_features=291, epochs=20, lr=3e-4,
                 hidden=32, dropout=0.3, warmup=None, pos_weight=50.0,
                 patience=5, seed=0, model=None, T=192):
    torch.manual_seed(seed)
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    m = (model or SeizureTCN(n_features=n_features, hidden=hidden,
                             dropout=dropout)).to(dev)
    warmup = warmup if warmup is not None else min(m.receptive_field, T // 2)
    print(f'device {dev} | warmup {warmup} | pos_weight {pos_weight} | '
          f'{len(dl_tr.dataset):,} seqs')

    lossf = nn.BCEWithLogitsLoss(
        reduction='none', pos_weight=torch.tensor(pos_weight, device=dev))
    opt = torch.optim.AdamW([p for p in m.parameters() if p.requires_grad],
                            lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.amp.GradScaler('cuda')
    mask = torch.zeros(T, device=dev); mask[warmup:] = 1.0

    best, best_state, bad = np.inf, None, 0
    for ep in range(epochs):
        m.train(); tot = n = 0
        for xb, yb in dl_tr:
            xb, yb = xb.to(dev, non_blocking=True), yb.to(dev, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda'):
                l = (lossf(m(xb), yb) * mask).sum() / (mask.sum() * xb.shape[0])
            scaler.scale(l).backward()
            scaler.unscale_(opt)                       # clip UNSCALED gradients
            nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            scaler.step(opt); scaler.update()
            tot += l.item(); n += 1
        sched.step()

        if dl_va is not None:
            m.eval(); vt = vn = 0
            with torch.no_grad():
                for xb, yb in dl_va:
                    xb, yb = xb.to(dev), yb.to(dev)
                    vt += ((lossf(m(xb), yb) * mask).sum() /
                           (mask.sum() * xb.shape[0])).item()
                    vn += 1
            vl = vt / vn
            flag = ''
            if vl < best - 1e-4:
                best, bad = vl, 0
                best_state = {k: v.detach().clone() for k, v in m.state_dict().items()}
                flag = ' *'
            else:
                bad += 1
            print(f'  ep {ep:2d}  train {tot/n:.4f}  val {vl:.4f}{flag}')
            if bad >= patience:
                print(f'  early stop at {ep}'); break
        else:
            print(f'  ep {ep:2d}  train {tot/n:.4f}')

    if best_state: m.load_state_dict(best_state)
    return m, warmup

In [ ]:
# Scoring helper

@torch.no_grad()
def score_dataset(model, ds, n_rows, warmup, T=192, bs=256):
    model.eval()
    lg, rows = [], []
    for i in range(0, len(ds.starts), bs):
        ix = ds.starts[i:i+bs]
        xb = torch.stack([torch.from_numpy(ds.X[s:s+T].T.copy()) for s in ix]).cuda()
        with torch.amp.autocast('cuda'):
            lg.append(model(xb).float().cpu().numpy())
        rows.append(np.stack([np.arange(s, s+T) for s in ix]))
    return sequences_to_window_scores(np.concatenate(lg), np.concatenate(rows),
                                      n_rows, warmup)

### 14.14 Per-subject sweep

Random initialization, not the pretrained trunk. Protocol matches the
classical calibration curve so the comparison in 14.4 is direct.

Checkpoint every model. The original sweep was not saved and had to be
partly regenerated.

In [ ]:
# Full sweep

def tcn_subject(subj, n_cal=5, T=192, stride_tr=24, stride_ev=48,
                epochs=40, lr=1e-4, hidden=32, dropout=0.3,
                pos_weight=50.0, target_fa=2.0, max_duty=0.01, seed=0):
    """Per-subject TCN, random init (pretraining showed NEGATIVE transfer).

    Protocol matches the classical calibration curve: seizure-bearing records
    only, first n_cal for training + threshold calibration, last 3 held out.
    """
    torch.manual_seed(seed)
    d = load_subjects([subj], subsample=1.0)
    sz = sorted(d.loc[d.y == 1, 'record'].unique())
    if len(sz) < n_cal + 3:
        n_cal = max(1, len(sz) - 3)
    tr = d[d.record.isin(sz[:n_cal])].reset_index(drop=True)
    te = d[d.record.isin(sz[-3:])].reset_index(drop=True)
    if tr.y.sum() == 0 or te.y.sum() == 0:
        print(f'{subj}: SKIP - a split has no seizures'); return None

    ds_tr = SeqDataset(tr, FEAT, T=T, stride=stride_tr)
    ds_ca = SeqDataset(tr, FEAT, T=T, stride=stride_ev, mu=ds_tr.mu, sd=ds_tr.sd)
    ds_te = SeqDataset(te, FEAT, T=T, stride=stride_ev, mu=ds_tr.mu, sd=ds_tr.sd)
    dl = DataLoader(ds_tr, batch_size=64, shuffle=True, drop_last=True)

    m = SeizureTCN(n_features=len(FEAT), hidden=hidden, dropout=dropout).cuda()
    wu = min(m.receptive_field, T // 2)
    lossf = nn.BCEWithLogitsLoss(reduction='none',
                                 pos_weight=torch.tensor(pos_weight, device='cuda'))
    opt = torch.optim.AdamW(m.parameters(), lr=lr)
    mask = torch.zeros(T, device='cuda'); mask[wu:] = 1.0

    for ep in range(epochs):
        m.train()
        for xb, yb in dl:
            xb, yb = xb.cuda(), yb.cuda()
            opt.zero_grad(set_to_none=True)
            l = (lossf(m(xb), yb) * mask).sum() / (mask.sum() * xb.shape[0])
            l.backward(); opt.step()

    s_ca, ok_ca = score_dataset(m, ds_ca, len(tr), wu, T=T)
    s_te, ok_te = score_dataset(m, ds_te, len(te), wu, T=T)
    mc = tr.loc[ok_ca, list(META_COLS)].reset_index(drop=True)
    mt = te.loc[ok_te, list(META_COLS)].reset_index(drop=True)

    q  = select_quantile(s_ca[ok_ca], mc, target_fa, max_duty)      # never test data
    wm = window_metrics(mt.y.to_numpy(), s_te[ok_te])
    em = event_metrics(mt, s_te[ok_te], float(np.quantile(s_te[ok_te], q)))

    row = {'subject': subj, 'n_cal': n_cal,
           'n_test_events': mt.loc[mt.y == 1, 'seizure_event'].nunique(),
           'lift': wm['auprc_lift'], 'auroc': wm['auroc'], 'q': q,
           'sens': em['sensitivity_event'], 'fa': em['fa_per_hour'],
           'duty': em['alarm_duty'], 'lat': em['latency_median_s']}
    print(f'{subj}: lift {row["lift"]:6.2f}x  auroc {row["auroc"]:.3f}  '
          f'sens {row["sens"]:.2f} @ {row["fa"]:5.2f} FA/h  '
          f'lat {row["lat"]:5.1f}s  duty {row["duty"]:.4f}  ({row["n_test_events"]} ev)')

    sweep = []
    for qq in (0.95, 0.98, 0.99, 0.995, 0.999):
        e = event_metrics(mt, s_te[ok_te], float(np.quantile(s_te[ok_te], qq)))
        sweep.append({'subject': subj, 'q': qq, 'sens': e['sensitivity_event'],
                      'fa': e['fa_per_hour'], 'lat': e['latency_median_s'],
                      'duty': e['alarm_duty']})

    del ds_tr, ds_ca, ds_te, m; gc.collect(); torch.cuda.empty_cache()
    return row, pd.DataFrame(sweep)

## 14.2 Compute budget - initial estimate was WRONG

An early estimate of ~93k MACs/timestep assumed **one** conv per layer. A standard
TCN residual block has **two** convs plus a 1x1 downsample.

| block | MACs/timestep |
|---|---|
| block 0 (291->64) | **86,784** |
| blocks 1-3 (64->64 each) | 24,576 |
| **total** | **160,576** |

**25% of the ~650k feature-extraction budget, not 14%.** Block 0 alone is 54% of
the model.

| configuration | MACs | params | % of budget |
|---|---|---|---|
| 291 feat, hidden=64 | 160,576 | 161,665 | 25% |
| 291 feat, hidden=32 | 58,784 | 59,329 | 9% |
| **32 feat (4ch x 8), hidden=32** | **24,608** | **25,121** | **4%** |

6.5x reduction at an unchanged 61-timestep receptive field.

-> **The second, sharper motivation for the feature-budget curve: input dimension
dominates the model itself, not just extraction.**

## 14.3 Within-subject: 18.8x, and data-limited

chb15 on its own records, three configurations:

| T | stride | warmup | lift |
|---|---|---|---|
| 96 | 48 | none (bug) | 18.54x |
| 96 | 48 | 48 | 18.52x |
| 192 | 48 | 61 | 18.82x |

All land at 18.5-18.8x with **validation bottoming at epoch 1-2**. Neither
sequence length nor warmup masking moved it. AUROC 0.978-0.980 - ranking is
excellent, the loss is in precision near the boundary.

The binding constraint is **~17 seizure events**. No architectural change creates
more. Against the classical within-subject ceiling of 36-56x, the TCN reaches
roughly half.

## 14.4 Cross-subject: BELOW CHANCE

Population pretraining on 22 groups, chb15 held out entirely:
**342,519 windows, 178 events** - 10.5x chb15 alone.

Training was healthy in a way per-subject training never was: validation improved
on 19 of 20 epochs, then converged over 60 total epochs to 0.0039 with no
overfitting.

On chb15's held-out records:

```
AUROC 0.411      AUPRC lift 0.79x
```

**Below chance.** The model ranks ictal windows *lower* than interictal for this
subject.

**Pipeline correctness confirmed:** scoring chb10 (which *was* in training) gives
AUROC 0.99996, lift 94.2x. Memorization, not generalization - but it proves the
scoring path and sign convention are right.

**Negating the scores gives lift 4.35x.** The trunk learned real structure that is
**systematically inverted** for chb15 - a more precise claim than "it failed", and
it matches the r = 0.106 feature-level finding at the representation level.

## 14.5 Fine-tuning: head-only fails, full partially recovers

| approach | trainable | lift |
|---|---|---|
| population only | - | 0.79x |
| + head fine-tune (LR 1e-3) | 33 | 0.72x, *falling* to 0.67x |
| + head fine-tune (LR 1e-5) | 33 | 0.72x, flat |
| + **full fine-tune (LR 1e-4)** | 59,329 | **14.28x**, still rising |
| chb15-only from scratch | 59,329 | 18.8x |

**33 parameters cannot recombine an inverted representation into a correct one.**
The head is a 1x1 conv over 32 channels - it can rescale and mix, not repair.

### A hypothesis tested and rejected

*Was the head collapsing the output bias to satisfy the loss?* No. **AUROC is
rank-based and therefore exactly invariant to an additive shift.** Verified -
shifting every logit by -100 left AUROC unchanged at 0.32014 to all printed
digits; negating gave exactly 1 - 0.32014.

The damage came from the 32 **weights** overfitting 408 ictal windows.

In [ ]:
# the check that rejected the bias hypothesis
s, ok = score_dataset(m_ft, ds_te, len(te15), warmup)
print('as-is   :', window_metrics(yte[ok],  s[ok])['auroc'])
print('shifted :', window_metrics(yte[ok],  s[ok] - 100)['auroc'])   # identical
print('negated :', window_metrics(yte[ok], -s[ok])['auroc'])         # 1 - auroc
print('negated lift:', window_metrics(yte[ok], -s[ok])['auprc_lift'])

## 14.6 B vs C - does pretraining contribute anything?

Full fine-tuning from the **pretrained** trunk vs from **random** init. Identical
architecture, data, learning rate - only initialization differs.

- **B ~ C at plateau** -> pretraining contributed nothing; the weights are an
  initialization, not transferred knowledge. That is the negative result.
- **B > C** -> useful initialization despite no direct transfer.
- **B plateaus in far fewer epochs** -> pretraining shortens *calibration time*,
  a genuine deployment benefit even at equal final performance.

Reference point: **18.8x**, the chb15-only model. Below that means an hour of GPU
pretraining underperforms three minutes of per-subject training.

In [ ]:
def finetune(model, lr, tag, epochs=20, freeze=False):
    m = copy.deepcopy(model).cuda()                       # device-safe
    if freeze:
        m.freeze_trunk()
        for p in m.head.parameters(): p.requires_grad = True
    else:
        for p in m.parameters(): p.requires_grad = True
    tp = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'\n{tag}: {tp:,} trainable, lr={lr}')

    lossf = nn.BCEWithLogitsLoss(reduction='none',
                                 pos_weight=torch.tensor(50.0, device='cuda'))
    opt = torch.optim.AdamW([p for p in m.parameters() if p.requires_grad], lr=lr)
    mask = torch.zeros(192, device='cuda'); mask[warmup:] = 1.0

    hist = []
    for ep in range(epochs):
        m.train()
        for xb, yb in dl_ft:
            xb, yb = xb.cuda(), yb.cuda()
            opt.zero_grad(set_to_none=True)
            l = (lossf(m(xb), yb) * mask).sum() / (mask.sum() * xb.shape[0])
            l.backward(); opt.step()
        if ep % 4 == 0 or ep == epochs - 1:
            sc, o = score_dataset(m, ds_te, len(te15), warmup)
            w = window_metrics(yte[o], sc[o])
            hist.append((ep, w['auprc_lift'], w['auroc']))
            print(f'  ep {ep:2d}  lift {w["auprc_lift"]:6.2f}x  auroc {w["auroc"]:.3f}')
    return m, hist


m_b2, hist_b = finetune(m_pop, 1e-4, 'B: pretrained init', epochs=40)
m_c,  hist_c = finetune(SeizureTCN(n_features=len(FEAT), hidden=32, dropout=0.3),
                        1e-4, 'C: random init', epochs=40)

## 14.7 Bugs this section produced - all now guarded

**RAM crash (D50).** `build_sequences` materialized every sequence: at T=128,
stride=8 that is **16 copies** of each feature value. chb09 alone produced 4.5 GB
and the normalisation step doubled it. Fixed with a size guard, in-place
normalisation, and a streaming `SeqDataset` that indexes into one flat matrix -
0.40 GB for the whole population.

**`pos_weight = 1024.5` (D51).** The theoretical value at chb09's 0.098%
imbalance produced catastrophic memorization: train loss 0.0135 while validation
rose to 35.5. The loss surface becomes dominated by a handful of samples. Capped
at 50.

**Trained on zero seizures (D52).** `recs[:5]` picked five records from chb15's
40 - but only **14 contain seizures**. The fine-tuning run trained on 0.000%
ictal and the loss decreased smoothly while learning nothing. Now selects from
`d15.loc[d15.y==1, 'record'].unique()` with an assert.

**`warmup=None` silently disabled the loss mask.** `train_tcn` returned `None`
because the default was never resolved, so no timesteps were excluded. Measured
impact: 18.54x -> 18.52x, i.e. negligible - but it would not have been on a
shorter T.

**Evaluating on loss alone (D53).** Head fine-tuning showed loss falling
53.7 -> 18.3 while AUROC fell 0.344 -> 0.320. Loss is not comparable across
`pos_weight` settings and is not the objective.

## 14.8 Outstanding

**The TCN has never been scored with `event_metrics`.** Every number above is
window-level AUPRC. Sensitivity, FA/h, alarm duty and detection latency - what
the write-up reports and what the classical baseline is measured on - do not
exist for it yet. **No apples-to-apples comparison is currently possible.**

**Quantile threshold transfer (D44) has not been applied to any TCN result.**

---
# 15. Streaming fixed-point port

## 15.1 Streaming, not recompute (D55)

Re-running a 192-step window per new timestep does **192x the necessary work**.
A causal dilated conv at *t* needs only `(k-1)*dilation` past activations, so
each layer keeps a ring buffer and computes one output column.

| | per-window MACs | activation cache |
|---|---|---|
| naive recompute | 11,286,528 | none |
| **streaming** | **58,784** | 2,953 values = **5.8 KB int16** |

The input buffer alone (291 features x 3 taps = 873 values) is **30% of the
cache** - a third argument for feature reduction.

**The budget is 1000 ms, not 15.6 ms (D56).** An earlier framing conflated two
rates: the TCN runs once per *window* (1 s stride), only the filter runs per
sample. Measured 0.175 ms p50 in interpreted NumPy - 0.02% of budget. Compute
was never the constraint; **energy is**.

| validation | result |
|---|---|
| streaming vs batch, synthetic | 2.13e-07 |
| streaming vs batch, trained model | 3.72e-07 |
| **streaming vs batch, real EEG features** | **1.53e-06** |
| reset isolation (two runs after reset) | **0.000e+00** |

### 15.5 Streaming inference

A causal dilated conv at timestep t needs only `(k-1)*dilation` past
activations, so each layer keeps a ring buffer and computes one output
column. 58,784 MACs against 11,286,528 for a full recompute.

In [ ]:
# Streaming Inference

from scipy.special import erf
from math import sqrt


class StreamingConv1d:
    """One causal dilated conv, evaluated one timestep at a time.

    Ring buffer holds (k-1)*dilation + 1 past inputs. Each step advances the
    buffer and produces exactly one output column - instead of re-running a
    192-step window, which does 192x the necessary work.
    """
    def __init__(self, conv):
        self.k, self.d = conv.kernel_size[0], conv.dilation[0]
        self.span = (self.k - 1) * self.d + 1
        self.W = conv.weight.detach().cpu().numpy().astype(np.float64)
        self.b = (conv.bias.detach().cpu().numpy().astype(np.float64)
                  if conv.bias is not None else np.zeros(self.W.shape[0]))
        self.in_ch, self.out_ch = self.W.shape[1], self.W.shape[0]
        self.reset()

    def reset(self):
        """Zeros - the batch path left-pads with zeros, so this is exact."""
        self.buf = np.zeros((self.in_ch, self.span), np.float64)

    def step(self, x_t):
        self.buf = np.roll(self.buf, -1, axis=1)
        self.buf[:, -1] = x_t
        taps = self.buf[:, ::self.d][:, -self.k:]        # oldest first
        return np.einsum('oik,ik->o', self.W, taps) + self.b

    @property
    def cache_size(self):
        return self.in_ch * self.span


class StreamingTCN:
    """Incremental SeizureTCN. Mirrors the batch architecture exactly;
    dropout is identity at inference."""
    def __init__(self, model):
        model = model.eval().cpu()
        self.blocks = []
        for b in model.blocks:
            self.blocks.append({
                'c1': StreamingConv1d(b.c1),
                'c2': StreamingConv1d(b.c2),
                'down':   (b.down.weight.detach().cpu().numpy().astype(np.float64)[:, :, 0]
                           if b.down is not None else None),
                'down_b': (b.down.bias.detach().cpu().numpy().astype(np.float64)
                           if b.down is not None else None)})
        self.head_W = model.head.weight.detach().cpu().numpy().astype(np.float64)[:, :, 0]
        self.head_b = model.head.bias.detach().cpu().numpy().astype(np.float64)
        self.reset()

    def reset(self):
        for b in self.blocks:
            b['c1'].reset(); b['c2'].reset()

    @staticmethod
    def _gelu(x):                       # exact erf form, matching torch default
        return 0.5 * x * (1.0 + erf(x / sqrt(2.0)))

    def step(self, x_t):
        x = np.asarray(x_t, np.float64)
        for b in self.blocks:
            res = x if b['down'] is None else b['down'] @ x + b['down_b']
            y = self._gelu(b['c1'].step(x))
            y = self._gelu(b['c2'].step(y))
            x = self._gelu(y + res)
        return float((self.head_W @ x + self.head_b)[0])

    @property
    def cache_values(self):
        return sum(b['c1'].cache_size + b['c2'].cache_size for b in self.blocks)

    def report(self):
        macs = 0
        for b in self.blocks:
            macs += b['c1'].in_ch * b['c1'].out_ch * b['c1'].k
            macs += b['c2'].in_ch * b['c2'].out_ch * b['c2'].k
            if b['down'] is not None:
                macs += b['down'].shape[0] * b['down'].shape[1]
        macs += self.head_W.shape[1]
        cv = self.cache_values
        print(f'  MACs / timestep  {macs:,}   (recompute would be {macs*192:,})')
        print(f'  activation cache {cv:,} values '
              f'({cv*2/1024:.1f} KB int16, {cv*4/1024:.1f} KB float32)')
        return macs, cv

### 15.6 The contract

The incremental path must reproduce the batch forward pass. This is the
same contract the causal filter satisfies in 6.1. Without it, training in
batch and deploying incrementally are two different models and nothing
would reveal it.

In [ ]:
# The validation

def test_stream_equals_batch(model, F=291, T=64, seed=0, tol=1e-6):
    """Incremental path must reproduce the batch forward pass.

    Same contract the causal filter satisfies. Without it, training in batch and
    deploying incrementally are two different models, and nothing would reveal it.
    """
    torch.manual_seed(seed)
    x = torch.randn(1, F, T)
    with torch.no_grad():
        batch = model.eval().cpu()(x)[0].numpy().astype(np.float64)
    st = StreamingTCN(model)
    stream = np.array([st.step(x[0, :, t].numpy()) for t in range(T)])
    d = np.abs(batch - stream).max()
    print(f'  max |batch - stream| : {d:.3e}   (tol {tol:g})')
    assert d < tol, f'streaming diverges by {d:.3e}'
    print('  PASS')
    return d


def test_reset_isolation(model, F=291, T=32, seed=1):
    """Reset must fully clear state - two runs after reset must be identical."""
    torch.manual_seed(seed)
    x = torch.randn(1, F, T)
    st = StreamingTCN(model)
    a = np.array([st.step(x[0, :, t].numpy()) for t in range(T)])
    st.reset()
    b = np.array([st.step(x[0, :, t].numpy()) for t in range(T)])
    d = np.abs(a - b).max()
    print(f'  max |run1 - run2| after reset : {d:.3e}')
    assert d < 1e-12, 'reset does not clear state'
    print('  PASS')
    return d


def benchmark_streaming(model, F=291, n=1500, seed=0):
    """The TCN runs once per WINDOW (1 s stride) -> 1000 ms budget, not the
    15.6 ms packet budget that governs the filter."""
    import time
    rng = np.random.default_rng(seed)
    st = StreamingTCN(model); xs = rng.standard_normal((n, F))
    for i in range(50): st.step(xs[i])
    ts = []
    for i in range(n):
        t0 = time.perf_counter(); st.step(xs[i]); ts.append((time.perf_counter()-t0)*1e3)
    ts = np.array(ts)
    print(f'  p50 {np.percentile(ts,50):.3f} ms | p99 {np.percentile(ts,99):.3f} ms '
          f'| {np.percentile(ts,99)/1000*100:.3f}% of 1000 ms budget')
    return ts


m_ref = SeizureTCN(n_features=len(FEAT), hidden=32, dropout=0.0)
print('stream == batch:');   test_stream_equals_batch(m_ref, F=len(FEAT), T=64)
print('\nreset isolation:'); test_reset_isolation(m_ref, F=len(FEAT))
print('\nfootprint:');       StreamingTCN(m_ref).report()
print('\nwall clock:');      benchmark_streaming(m_ref, F=len(FEAT))

In [ ]:
m_ref_rand = SeizureTCN(n_features=291, hidden=32, dropout=0.0)
print('stream == batch:');   test_stream_equals_batch(m_ref_rand, F=291, T=64)
print('\nreset isolation:'); test_reset_isolation(m_ref_rand, F=291)
print('\nfootprint:');       StreamingTCN(m_ref_rand).report()
print('\nwall clock:');      benchmark_streaming(m_ref_rand, F=291)

### 15.7 Activation ranges from real data

Q15 scaling must come from measured dynamic range, not assumption.
Overflow is silent and catastrophic.

Measured: blocks max 3.9, head max 6.6, and `max` sits close to `p99.9`
everywhere, so there are no rare outliers wasting the scale.

In [ ]:
# Q15 - The approach here will catch any issues

def q15_stats(model, X_real, F, T=192):
    """Per-layer activation ranges from REAL data - Q15 scaling must come from
    measured dynamic range, not assumption. Overflow is silent and catastrophic."""
    m = model.eval().cpu()
    acts = {}
    hooks = []

    def mk(name):
        def hook(mod, inp, out):
            a = out.detach().numpy()
            acts.setdefault(name, []).append((float(np.abs(a).max()),
                                              float(np.percentile(np.abs(a), 99.9))))
        return hook

    for i, b in enumerate(m.blocks):
        hooks.append(b.c1.register_forward_hook(mk(f'block{i}.c1')))
        hooks.append(b.c2.register_forward_hook(mk(f'block{i}.c2')))
    hooks.append(m.head.register_forward_hook(mk('head')))

    with torch.no_grad():
        m(torch.from_numpy(X_real.T[None].astype(np.float32)))
    for h in hooks: h.remove()

    print(f'{"layer":14s} {"max|a|":>10s} {"p99.9":>10s} {"Q15 int bits":>13s}')
    out = {}
    for k, v in acts.items():
        mx = max(x[0] for x in v); p999 = max(x[1] for x in v)
        bits = int(np.ceil(np.log2(max(mx, 1e-9)))) + 1
        out[k] = dict(max=mx, p999=p999, int_bits=bits)
        print(f'{k:14s} {mx:10.3f} {p999:10.3f} {bits:13d}')

    # weights matter too - they scale independently of activations
    print(f'\n{"param":22s} {"max|w|":>10s}')
    for n, p in m.named_parameters():
        if 'weight' in n:
            print(f'{n:22s} {p.detach().abs().max().item():10.4f}')
    return out

In [ ]:
stats = q15_stats(m_c, Xs, len(FEAT))

### 15.8 Effective weights

Weight norm materializes `w = g * v/||v||` on attribute access. If a
checkpoint has damaged parametrizations, reconstruct from the raw
parameters instead.

In [ ]:
def wn_weight(mod):
    """Reconstruct weight-norm's effective weight: w = g * v / ||v||.
    torch normalizes over all dims except dim=0 (output channels)."""
    ps = dict(mod.named_parameters())
    g = ps['parametrizations.weight.original0']       # magnitude
    v = ps['parametrizations.weight.original1']       # direction
    n = v.detach().flatten(1).norm(dim=1).view(-1, 1, 1)
    return (g.detach() * v.detach() / n).numpy().astype(np.float64)

In [ ]:
Wf = {}
for name, mod in m_c.named_modules():
    if isinstance(mod, nn.Conv1d):
        w = (wn_weight(mod)
             if 'parametrizations.weight.original0' in dict(mod.named_parameters())
             else mod.weight.detach().numpy().astype(np.float64))
        Wf[name] = {'w': w,
                    'b': (mod.bias.detach().numpy().astype(np.float64)
                          if mod.bias is not None else np.zeros(w.shape[0])),
                    'k': mod.kernel_size[0], 'd': mod.dilation[0]}
        print(f'{name:18s} {str(w.shape):18s} max|w| {np.abs(w).max():.4f}')

### 15.9 Quantization: precision-optimal shifts overflow

The first pass shows the problem. Precision-optimal shifts of 17 to 18
put the worst-case accumulator at 33 to 36 bits, overflowing int32 on
every layer.

In [ ]:
def quantize_tensor(w, bits=15):
    """Symmetric per-tensor to int16 with a power-of-two scale, so
    dequantization is a shift rather than a multiply - that is what makes it
    cheap on a DSP with no hardware divider."""
    mx = float(np.abs(w).max())
    shift = int(np.floor(np.log2((2**bits - 1) / max(mx, 1e-12))))
    q = np.clip(np.round(w * (2.0**shift)), -32768, 32767).astype(np.int32)
    return q, shift


print(f'{"layer":18s} {"max|w|":>9s} {"shift":>6s} {"rel err":>10s}')
QW = {}
for name, v in Wf.items():
    q, sh = quantize_tensor(v['w'])
    rel = np.abs(q / 2.0**sh - v['w']).max() / np.abs(v['w']).max()
    QW[name] = {'q': q, 'shift': sh, 'b': v['b'], 'k': v['k'], 'd': v['d']}
    print(f'{name:18s} {np.abs(v["w"]).max():9.4f} {sh:6d} {rel:10.2e}')

ACT_BITS = 12                                    # Q3.12 activations
print(f'\n{"layer":18s} {"terms":>7s} {"worst |acc|":>13s} {"bits":>6s} {"int32":>9s}')
for name, v in QW.items():
    n_terms = Wf[name]['w'].shape[1] * Wf[name]['w'].shape[2]
    a_max = 8.0 if 'head' in name else 4.0
    worst = (np.abs(Wf[name]['w']).sum(axis=(1, 2)).max()
             * a_max * 2.0**v['shift'] * 2.0**ACT_BITS)
    bits = np.log2(max(worst, 1.0))
    print(f'{name:18s} {n_terms:7d} {worst:13.3e} {bits:6.1f} '
          f'{"OK" if bits < 31 else "OVERFLOW":>9s}')

### 15.10 Reduce the shift to buy headroom

Five bits of weight precision bought six bits of accumulator headroom,
at a cost of about 1e-03 relative error. That is well under the 1e-02
where AUPRC would move.

The worst case assumes every input at maximum with all signs aligned to
the weights, which never occurs. Designing to it is still correct.

In [ ]:
# Reduce the weight shift.
# ACT_BITS comes from the previous cell (Q3.12 activations).

TARGET_BITS = 30                        # int32 with 1 bit of margin


def choose_shift(w, n_terms, a_max, act_bits=ACT_BITS, target=TARGET_BITS, bits=15):
    """Pick the largest weight shift that keeps the worst-case accumulator inside
    int32.

    Weight precision is cheap here (3e-05 rel err at shift 17) but accumulator
    overflow is silent and catastrophic, so spend precision to buy headroom.
    """
    mx = float(np.abs(w).max())
    sh_prec = int(np.floor(np.log2((2**bits - 1) / max(mx, 1e-12))))   # precision-optimal
    row_sum = float(np.abs(w).sum(axis=(1, 2)).max())
    # worst |acc| = row_sum * a_max * 2^sh * 2^act_bits  <=  2^target
    sh_safe = int(np.floor(target - np.log2(row_sum * a_max) - act_bits))
    return min(sh_prec, sh_safe), sh_prec, sh_safe


print(f'{"layer":18s} {"prec sh":>8s} {"safe sh":>8s} {"used":>5s} '
      f'{"acc bits":>9s} {"rel err":>10s}')
QW = {}
for name, v in Wf.items():
    w = v['w']
    a_max = 8.0 if 'head' in name else 4.0
    sh, sp, ss = choose_shift(w, w.shape[1] * w.shape[2], a_max)
    q = np.clip(np.round(w * 2.0**sh), -32768, 32767).astype(np.int32)
    rel = np.abs(q / 2.0**sh - w).max() / np.abs(w).max()
    worst = np.abs(w).sum(axis=(1, 2)).max() * a_max * 2.0**sh * 2.0**ACT_BITS
    QW[name] = {'q': q, 'shift': sh, 'b': v['b'], 'k': v['k'], 'd': v['d']}
    print(f'{name:18s} {sp:8d} {ss:8d} {sh:5d} {np.log2(worst):9.1f} {rel:10.2e}')

### 15.11 GELU by lookup table

The only transcendental in the network. Your architecture makes 384 GELU
evaluations per timestep against 58,784 MACs. At roughly 100 cycles for
an erf-based GELU on a Cortex-M4 that is 65% on top of the MAC budget;
a table read plus a lerp is about 8 integer ops, or 5%.

The table also needs no FPU, is constant time, and has a known error
bound rather than an inherited one.

**The table range must cover the measured activation range.** Clipping is
silent: at x=12 with a range of +/-8 the error is 4.0.

In [ ]:
# Q15 forward pass against float64 on real data

def gelu_lut(n=512, lo=-8.0, hi=8.0):
    """GELU as a lookup table with linear interpolation.

    It is the only transcendental in the network. On a DSP with no FPU an
    erf-based GELU is far more expensive than one table read plus a lerp, and
    the table error is what will dominate quantization error if it is coarse.
    """
    xs = np.linspace(lo, hi, n)
    ys = 0.5 * xs * (1.0 + erf(xs / sqrt(2.0)))
    return xs, ys, (hi - lo) / (n - 1)


LUT_X, LUT_Y, LUT_STEP = gelu_lut()


def gelu_q(x, lut_x=LUT_X, lut_y=LUT_Y, step=LUT_STEP):
    xi = np.clip((x - lut_x[0]) / step, 0, len(lut_x) - 1.0001)
    i = xi.astype(np.int32); f = xi - i
    return lut_y[i] * (1 - f) + lut_y[i + 1] * f


# how much does the table alone cost?
t = np.linspace(-8, 8, 20000)
exact = 0.5 * t * (1 + erf(t / sqrt(2.0)))
print(f'GELU LUT (512 pts): max abs err {np.abs(gelu_q(t) - exact).max():.2e}')

### 15.12 Plain-conv deployment architecture

Weight norm does not survive export and its parametrization machinery
broke three separate times. This is the deployment form.

In [ ]:
# Skip the removal entirely. Define a plain-conv architecture and load

class PlainTCNBlock(nn.Module):
    """Identical to TCNBlock but without weight-norm. Weight-norm is a
    training-time convenience that does not survive to deployment, and its
    parametrization machinery has now broken three times."""
    def __init__(self, in_ch, out_ch, kernel_size, dilation, dropout=0.0):
        super().__init__()
        self.p = (kernel_size - 1) * dilation
        self.c1 = nn.Conv1d(in_ch, out_ch, kernel_size, dilation=dilation)
        self.c2 = nn.Conv1d(out_ch, out_ch, kernel_size, dilation=dilation)
        self.act, self.drop = nn.GELU(), nn.Dropout(dropout)
        self.down = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else None

    def forward(self, x):
        r = x if self.down is None else self.down(x)
        y = self.drop(self.act(self.c1(nn.functional.pad(x, (self.p, 0)))))
        y = self.drop(self.act(self.c2(nn.functional.pad(y, (self.p, 0)))))
        return self.act(y + r)


class PlainSeizureTCN(nn.Module):
    def __init__(self, n_features=291, hidden=32, kernel_size=3,
                 dilations=(1, 2, 4, 8), dropout=0.0):
        super().__init__()
        ch = [n_features] + [hidden] * len(dilations)
        self.blocks = nn.ModuleList([
            PlainTCNBlock(ch[i], ch[i+1], kernel_size, d, dropout)
            for i, d in enumerate(dilations)])
        self.head = nn.Conv1d(hidden, 1, 1)
        self.receptive_field = 1 + 2 * (kernel_size - 1) * sum(dilations)

    def forward(self, x):
        for b in self.blocks:
            x = b(x)
        return self.head(x).squeeze(1)

In [ ]:
m_ref = PlainSeizureTCN(n_features=len(FEAT), hidden=32).eval().cpu()
with torch.no_grad():
    for name, mod in m_ref.named_modules():
        if isinstance(mod, nn.Conv1d):
            mod.weight.copy_(torch.from_numpy(Wf[name]['w']).float())
            mod.bias.copy_(torch.from_numpy(Wf[name]['b']).float())

for name, mod in m_ref.named_modules():
    if isinstance(mod, nn.Conv1d):
        print(f'{name:18s} max|w| {mod.weight.abs().max():.4f}')

## 15.2 Q15 quantization - the accumulator is the constraint (D57)

Weights int16 at **per-layer power-of-two scales**, so dequantization is a shift
not a multiply. Per-layer because `blocks.0.c1` peaks at |w| = 0.0637 while later
layers reach 0.13.

**Precision-optimal shifts overflow int32 on every layer.** At shift 17-18 the
worst-case accumulator is 33-36 bits; `blocks.0.c1` sums 873 products and is
worst at 35.9.

Fix: **spend weight precision to buy accumulator headroom.**

| layer | prec-optimal | safe | used | acc bits | rel err |
|---|---|---|---|---|---|
| `blocks.0.c1` | 18 | 12 | 12 | 29.9 | 1.92e-03 |
| `blocks.0.down` | 18 | 12 | 12 | 29.3 | 1.34e-03 |
| all others | 17 | 13 | 13 | ~29.5 | ~4.6e-04 |

Five bits of precision bought six bits of headroom, at ~1e-03 relative error -
well under the ~1e-02 where AUPRC would move.

The worst case assumes every input at maximum with all signs aligned to the
weights, which never occurs. Designing to it is still correct: **overflow is
silent and produces plausible garbage.**

**GELU by 512-point LUT (D58):** 9.77e-05 max error, negligible against the
1.9e-03 weight error and far cheaper than erf on a DSP with no FPU. It is the
only transcendental in the network.

> **Drop weight-norm from the deployment architecture (D59).** It reparameterizes
> the optimization and changes nothing about the trained function, but its
> parametrization machinery broke three separate times and does not survive
> export. `PlainSeizureTCN` is the deployment form; recover effective weights via
> `w = g * v/||v||` if a parametrized checkpoint is all you have.

## 15.3 Q15 costs ZERO detection performance (D60)

Full held-out set - 9,783 windows, 7 events - scored through both paths using the
**same quantile** for each (the deployable threshold rule, D44):

| metric | float64 | Q15 | delta |
|---|---|---|---|
| AUPRC lift | 15.5997 | 15.6019 | +0.0022 |
| AUROC | 0.9902 | 0.9902 | 0.0000 |
| sens @ q=0.95 | 0.8571 | 0.8571 | **0.0000** |
| FA/h @ q=0.95 | 2.3501 | 2.3501 | **0.0000** |
| duty @ q=0.95 | 0.0015 | 0.0015 | **0.0000** |
| latency @ q=0.95 | 9.5 s | 9.5 s | 0.0 |
| sens @ q=0.98 | 0.8571 | 0.8571 | **0.0000** |
| latency @ q=0.98 | 24.0 s | 22.5 s | **-1.5 s** |
| sens @ q=0.99 | 0.7143 | 0.7143 | **0.0000** |

**Sensitivity, FA/h and alarm duty are identical at every operating point.** The
only difference anywhere is 1.5 s of median latency at q=0.98, and it went
*down* - one borderline window crossing threshold earlier, not degradation.

Max score error 0.0158 against a range of 11.2 (**0.14%**), rank correlation
0.999999. Because the detector thresholds on a quantile, a uniform offset is
free; only reordering would hurt.

**Predicting is not measuring.** The correlation predicted no change; the
detection metrics confirmed it.

## 15.4 Deployment footprint

| component | size |
|---|---|
| TCN weights (int16) | 57.9 KB |
| activation cache (int16) | 5.8 KB |
| feature ring buffer (18 ch x 512, int16) | 18.4 KB |
| filter state | negligible |
| **total** | **~82 KB** |

Fits flash on any Cortex-M4, SRAM on an M7. Random forest at 100 trees is
~340 KB and does **not** fit. Logistic regression is 1.2 KB.

Smaller configs, same 61-timestep receptive field:

| config | params | int8 | MACs/step |
|---|---|---|---|
| 291 feat, h=32 *(deployed)* | 59,329 | 57.9 KB | 58,784 |
| 32 feat, h=32 | 25,121 | 24.5 KB | 24,608 |
| 32 feat, h=16 | 7,713 | 7.5 KB | 7,440 |

**The 32-feature configurations are untested.** That is the feature-budget curve.

---
# 16. Immediate next steps

Ordered by what unblocks the most, not by what is most interesting.

## 1. Email Roy - today

Attach the **manuscript draft**, not a summary. Three specific asks:

- **Is this preprint-worthy, and where?** He will know whether it fits
  *J. Neural Engineering*, *IEEE TBME*, or a workshop.
- **Can you or Aazhang sponsor CRC access?** Frame it with the number: the
  23-subject extension is ~17 GPU-hours, infeasible on Colab.
- **Does the r = 0.106 result connect to SP-BAND?** Whether patient-specificity
  also appears in aperiodic parameters links independent work to the lab's.

Twenty minutes, and it potentially unblocks both the compute and the faculty
acknowledgment - the two things that most change how this reads.

## 2. Render figures 1-4 (Section 13.3)

Code and results exist for all of them. **One hour.** This is the gap between a
project that is impressive to *read* and one that is impressive to *skim* - and
skimming is what actually happens.

## 3. Preregister the 23-subject extension - BEFORE running it

Everything so far is retrospective. Specify the protocol, primary metric,
operating point and exclusion rules. Expect writing it to surface at least two
choices you had not realised were choices; that is the value.

## 4. The 23-subject run

One seed for the 17 untested subjects, three seeds retained on the six already
done - ~6 h, defensible given event sensitivity had **zero variance** across
seeds. All three seeds if CRC access comes through.

> **Checkpoint every model.** The six-subject sweep was not saved and had to be
> partly regenerated.

## 5. Study section 4 of the guide while things run

Ben-David et al. (2010) is one paper and it is the largest gap between what has
been measured and what can be claimed. The bound decomposes target error into
source error, distributional divergence, and **lambda** - the error of the best joint
hypothesis. Alignment methods reduce the divergence term; **nothing reduces lambda**.
The r = 0.106 result is evidence lambda is large, which is exactly the case the theory
says cannot be fixed by reweighting or feature alignment. That is why
normalization recovered only 4%.

---

## Explicitly deferred

KiCad, Riemannian recentering, the C port, the feature-budget curve. All
worthwhile, none blocking, and all of them compete with finishing what is
started.

**Riemannian** is the strongest of these and remains the one untried method that
targets a *rotation* of the discriminative direction rather than a shift or
scale. Note it now competes against a *working* detector - it would have to beat
0.86-1.00 sensitivity at 0.4-1.8 FA/h, not merely improve on chance.

---

## Do not report accuracy

0.339% positive rate -> a null model scores 99.66%. Report AUPRC **against its
baseline**, event-level sensitivity at fixed FA/h, `alarm_duty`, and detection
latency. **Median** across folds, never the mean alone (D26, D32).

**Two distinct latencies.** *Compute*: per packet for the filter, per window for
the TCN - 15.6 ms and 1000 ms budgets respectively (D56). *Detection*: onset to
alarm, floored at 4 s by window length plus the 3-of-5 persistence rule.